# SOLA 4237 Assignment 2 - Household Battery Energy Storage System (BESS) Optimisation

### Site: `23W0158` - `NSW Thredbo, Alpine` - `2022` / `2023`

### SOLA4237 Consumer Energy Resources (CER) Operations and Integration

<div style="font-size: 1rem; font-weight: normal; line-height: 1.4;">
Student: <b>Bilal Parkar</b> &nbsp;|&nbsp; zID: <b>z5476566</b><br>
Prepared for: VPP-SPREE single-site optimisation division (retailer client review)
</div>

---

### Purpose and Context
This notebook holds **all analysis and optimisation code** behind the Assignment 2 report. The report must show that VPP-SPREE's single-site optimisation can deliver real, quantified benefit to a household across **flat, time-of-use (ToU) and wholesale pass-through tariffs**, and that it still works when **PV and price forecasts are imperfect**.

| # | Question the report must answer | Evidence built in |
|---|---|---|
| 1 | What are the site, data quality and tariff context? | Parts 1-4 |
| 2 | What battery (size, power, efficiency, limits, degradation) suits this site? | Parts 4, 5, 8 |
| 3 | What is the benchmark (no battery, simple self-consumption)? | Part 6 |
| 4 | What is the best possible operation on each tariff (perfect foresight)? | Part 7 |
| 5 | How good are our PV and price forecasts? | Parts 9, 10 |
| 6 | How much value is lost to uncertainty, and is PV or price the bigger driver? | Parts 11, 12 |
| 7 | Annual bill, self-consumption, savings, payback, VPP fee vs value | Part 13 |

### How this notebook is organised
- **Part 0** setup and constants. **Parts 1-3** load, check and prepare data. **Part 4** characterises the site. **Part 5** is the battery model. **Part 6** benchmarks. **Part 7** perfect-foresight optimiser. **Part 8** sizing and payback. **Parts 9-10** forecasts. **Part 11** 48-hour rolling-horizon scheduler with 5-minute execution. **Part 12** perfect vs uncertain comparison. **Part 13** techno-economics. **Part 14** discussion and recommendations. **Part 15** report figures. **Appendix A** AI disclosure and verification log. **Appendix B** reproducibility.
- Each task follows one pattern: *markdown explanation -> function -> test -> apply to site data -> findings cell*.
- Functions are tested on tiny synthetic samples (`assert`) before they touch site data.
- **Findings cells** (blockquotes) record what the data showed. Anything marked `TODO` is not finished.
- Set `SOLA_DATA_PATH` (or edit `DATA_PATH` in Part 0) and run top to bottom. Set `QUICK_RUN_DAYS` in Part 0 for a fast dry run of the slow parts (11-12).

### Required files
- Assignment CSV (two years of 5-minute data). Column names are mapped in one place (`COL` in Part 0): **check these against your CSV first.**
- AEMO pre-dispatch price forecasts for the site's region (only if you choose them over your own price model; see Part 10).

### Contents and status

| Part | Title | Status | Note |
|---|---|---|---|
| 0 | Setup and configuration | Done | Site metadata, paths, wholesale tariff settings (sourced) |
| 1 | Data loading and integrity | Done | Findings 1.2-1.6 and Part 1 summary written |
| 2 | Data cleaning | Done | Functions written by me; all tests and contract checks pass; Figure 2.1 |
| 3 | Tariffs and prices | In progress | 3.1-3.5 built; Finding 3.2-3.5 to write; network windows pending tutor |
| 4 | Site characterisation | Code done | Functions written; PV ~8.9 kWp; AC-coupled; findings 4.1-4.4 to write |
| 5 | Battery model | Skeleton ready | 5.1 product choice; 5.2-5.4 functions to write; capex to source |
| 6 | Benchmarks and billing | Skeleton | No battery, self-consumption |
| 7 | Perfect-foresight optimiser | Skeleton | LP written and tested; run on 3 tariffs |
| 8 | Sizing and payback | Skeleton | Capex assumptions need sources |
| 9 | PV forecasting | Skeleton | Own model vs commercial forecast |
| 10 | Price forecasting | Skeleton | AEMO pre-dispatch or own model |
| 11 | Rolling-horizon scheduler | Skeleton | 48 h hourly plan, 5 min execution |
| 12 | Perfect vs uncertain; PV vs price | Skeleton | Ablation study |
| 13 | Techno-economics and metrics | Skeleton | Master results table |
| 14 | Discussion and recommendations | Skeleton | Fill after results |
| 15 | Report figures | Skeleton | Planned figure list |
| A | AI disclosure and verification log | Skeleton | Fill as you work, not at the end |
| B | Reproducibility | Skeleton | Versions, seed, runtime |

#### Brief requirements tracker

| # | Requirement from the brief | Where | Status |
|---|---|---|---|
| 1 | Default self-consumption mode as benchmark | Part 6 | TODO (no-battery baseline already in Task 4.2: 54% self-consumption) |
| 2 | AC vs DC bus assumption, losses | Parts 4, 5 | In progress: options laid out (Task 4.4); my decision pending; losses in Part 5 |
| 3 | BESS size, power, SOC limits, temperature limits, degradation cost | Parts 4, 5, 8 | In progress: sizing evidence in Task 4.2 (shiftable median 7.4, 90th pct 10.7 kWh/day) |
| 4 | Optimal strategy for flat, ToU, wholesale | Part 7 | TODO (all three tariffs ready in `prices`, Task 3.4) |
| 5 | Wholesale retail price (network, fees, certificates, hedging) | Part 3 | **Done** (Tasks 3.2-3.3); network windows pending tutor advice, one setting in Part 0 |
| 6 | $25/month VPP fee on wholesale pass-through | Part 0, Part 13 | Set in Part 0 (`VPP_FEE_PER_MONTH`); applied in Part 13 |
| 7 | Zero export (curtail) when prices are negative | Part 7 | TODO (negative export prices already in `prices`) |
| 8 | PV forecast model trained on Year 1, compared with commercial forecast (MAE, RMSE, nRMSE) | Part 9 | TODO (caveat: irradiance is hourly-interpolated, Finding 1.6) |
| 9 | Price forecast (own model or AEMO pre-dispatch) | Part 10 | TODO |
| 10 | 48 h rolling horizon, hourly replanning, 5-minute execution without knowing current-interval price/solar | Part 11 | TODO |
| 11 | Settle on actual price, BESS flows, load and solar | Part 11 | TODO (actual 2023 AEMO prices sourced and aligned, Task 3.1) |
| 12 | Uncertainty vs perfect foresight; PV vs price influence | Part 12 | TODO |
| 13 | Annual bill, self-consumption, savings, payback | Parts 6, 8, 13 | TODO |
| 14 | AI disclosure with evidence of checking AI output | Appendix A | In progress: running log in `AI_LOG.md` |
| 15 | Missing and spurious data identified and cleaned, with justification | Parts 1, 2 | **Done** (Tasks 1.2-1.6, 2.1-2.4, Figure 2.1) |
| 16 | Time handling: timestamps, alignment, power vs energy | Parts 1, 3, 4 | **Done** for prices (interval-end stamps, Task 3.1); AEST clock check in Task 4.1 |
| 17 | Site load and solar analysis | Parts 1, 4 | In progress (Task 4.2) |
| 18 | Code shared via GitHub link | Appendix B | TODO: repo not yet on GitHub |


# Part 0 - Setup and Configuration

### Conventions used throughout (verify in Part 2)
- **Battery power** `batt_kW`: **positive = discharging**, **negative = charging** (same convention as Assignment 1).
- **Grid power** `grid_kW`: **positive = import**, **negative = export**.
- **Accounting identity:** `grid = load - (solar - curtailed) - battery`.
- **Power to energy:** `kWh = kW x dt_h` (`dt_h = 5/60` for raw data, `1` for hourly plans).
- **Prices** are in **$/kWh** inside the code. Wholesale data arrives in $/MWh, so divide by 1000.
- **Time:** timestamps are naive local time. State whether they are NEM time (fixed AEST, UTC+10) in Part 1 and align prices accordingly.

In [406]:
import os, io, warnings, dataclasses
from pathlib import Path
import requests
from dataclasses import dataclass, replace
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.optimize import linprog
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
RNG_SEED = 42
np.random.seed(RNG_SEED)

In [407]:
# ---------- Paths ----------
DATA_PATH = os.environ.get(
    "SOLA_DATA_PATH",
    "/Users/bilalp/Documents/SOLA4237/CER_Assignments/Assignment_2/SOLA/Data_NSW_Thredbo - Alpine_2022-2023_partial.csv",
)
PRICE_DIR = Path(DATA_PATH).parent / "data"     # AEMO price cache (Part 3), created on first run
AEMO_PRICE_URL = "https://aemo.com.au/aemo/data/nem/priceanddemand/PRICE_AND_DEMAND_{ym}_{region}.csv"
# ---------- Column map: the ONLY place raw CSV names appear ----------
COL = dict(
    time     = "timestamp",
    load     = "load_kW",
    solar    = "solar_output_kW",                        # measured PV (expected Year 1 only)
    solar_fc = "commercial_forecasted_solar_output_kW",  # commercial forecast (expected Year 2 only)
    irr      = "irradiance_W/m^2",
    temp     = "temperature_C",
    rrp      = "spot_price_AUD/kWh",                     # already $/kWh: do NOT divide by 1000
    flat_imp = "tariff_import_flat_AUD/kWh",
    flat_exp = "tariff_export_flat_AUD/kWh",
    tou_imp  = "tariff_import_tou_AUD/kWh",
    tou_exp  = "tariff_export_tou_AUD/kWh",
)

# ---------- Site metadata (from CER-data-generator screenshot) ----------
SITE_ID      = "NSW_Thredbo"
LOCATION     = "Thredbo, NSW (Alpine)"
NEM_REGION   = "NSW1"
LOAD_PROFILE = "23W0158 (scaled x29.1)"
PV_PR        = 0.85      # performance ratio
PV_GAMMA     = -0.004    # power temperature coefficient, per degC

# ---------- Time ----------
INTERVAL_MIN   = 5
DT_H           = INTERVAL_MIN / 60      # hours per interval: kWh = kW * DT_H
STEPS_PER_HOUR = 60 // INTERVAL_MIN

In [408]:
# ---------- Wholesale pass-through tariff (Part 3) ----------
# Network time windows: "assignment" = the site's own ToU windows (Finding 1.4, every day);
# "essential" = Essential Energy's network windows (weekdays only). Awaiting tutor advice: change here only.
NETWORK_WINDOWS = "assignment"
NETWORK_RATES = {   # Essential Energy BLNT3AL (LV Residential ToU, interval meter), c/kWh excl. GST
    "2022-23": dict(peak=16.1216, shoulder=12.0800, offpeak=4.8957),   # indicative NUoS schedule, Mar 2022
    "2023-24": dict(peak=17.6443, shoulder=12.6958, offpeak=5.0549),   # network price list, from 1 Jul 2023
}
LOSS_FACTOR = 1.034   # DLF x MLF, Essential NSLP (ACIL Allen for AER, DMO 2023-24, Table 4.33)
ENV_C_KWH   = 1.781   # LRET 7.27 + SRES 6.86 + NSW ESS 3.26 + PDRS 0.42 $/MWh (Tables 4.6-4.10)
FEES_C_KWH  = 0.157   # NEM management 1.15 + ancillary services 0.42 $/MWh (Tables 4.11-4.12)
HEDGE_C_KWH = 0.263   # prudential: AEMO 0.77 + hedge 1.86 $/MWh (Tables 4.16, 4.24)
GST = 0.10            # applied to wholesale imports so they compare with the GST-inclusive flat/ToU rates
VPP_FEE_PER_MONTH = 25.0   # $/month, wholesale pass-through only (applied in Part 13)

# Part 1 - Data Loading and Integrity

### Background: why check the timeline before anything else?
Every later calculation assumes one row per 5-minute interval. A missing or duplicated interval silently biases energy totals, and a clock shift (daylight saving) misaligns solar noon and price merges. "No nulls" is not the same as "complete". This dataset also mixes **two years with different columns populated** (measured solar in Year 1, commercial forecast in Year 2), so nulls in those columns are expected, not faults.

## Task 1.1 - Load the telemetry

In [409]:
def load_telemetry(path) -> pd.DataFrame:
    out = pd.read_csv(path, parse_dates=["timestamp"])
    return out.set_index("timestamp").sort_index()

In [410]:
"""Tests for Task 1.1"""
_csv = io.StringIO("timestamp,x\n2024-01-01 00:05:00,2\n2024-01-01 00:00:00,1\n")
_t = load_telemetry(_csv)
assert isinstance(_t.index, pd.DatetimeIndex)   # real datetimes, not strings
assert _t.index.is_monotonic_increasing          # sorted
assert _t["x"].tolist() == [1, 2]                # values travelled with their timestamps
print("Task 1.1 tests passed")

Task 1.1 tests passed


In [411]:
df = load_telemetry(DATA_PATH)
print(df.shape)
df.head()

(210240, 10)


,irradiance_W/m^2,temperature_C,solar_output_kW,commercial_forecasted_solar_output_kW,load_kW,spot_price_AUD/kWh,tariff_import_flat_AUD/kWh,tariff_export_flat_AUD/kWh,tariff_import_tou_AUD/kWh,tariff_export_tou_AUD/kWh
timestamp,,,,,,,,,,
2022-01-01 00:00:00,0.0,15.1,0.0,NaN,0.53,0.12477,0.3,0.05,0.15,0.05
2022-01-01 00:05:00,0.0,15.1,0.0,NaN,0.54,0.12486,0.3,0.05,0.15,0.05
2022-01-01 00:10:00,0.0,15.1,0.0,NaN,0.53,0.12602,0.3,0.05,0.15,0.05
2022-01-01 00:15:00,0.0,15.0,0.0,NaN,0.51,0.12493,0.3,0.05,0.15,0.05
2022-01-01 00:20:00,0.0,15.0,0.0,NaN,0.52,0.11354,0.3,0.05,0.15,0.05


## Task 1.2 - Timeline integrity and the two-year split
`check_timeline(df)` reports rows, expected rows, missing timestamps, duplicates. `split_years(df)` cuts the data at exactly one year after the first timestamp, so a financial-year dataset also splits correctly.

In [412]:
def check_timeline(df : pd.DataFrame, freq: str = "5min") -> dict:
    
    """Check the index is a complete, duplicate-free, regular time series.

    Compares the actual timestamps against an ideal grid from first to last
    timestamp at `freq`. Note: a complete timeline can still hold empty rows,
    so check nulls separately.
    """
    
    full = pd.date_range(df.index.min(), df.index.max(), freq = freq)

    return {
        "rows": len(df),
        "expected": len(full),
        "missing": len(full.difference(df.index)),
        "duplicates": int(df.index.duplicated().sum()),
        "monotonic": bool(df.index.is_monotonic_increasing),
        "start": df.index.min(),
        "end": df.index.max()
    }

In [413]:
"""Tests for Task 1.2a"""
# one missing interval (00:10)
_idx = pd.to_datetime(["2024-01-01 00:00", "2024-01-01 00:05", "2024-01-01 00:15"])
_r = check_timeline(pd.DataFrame({"a": [1, 2, 3]}, index=_idx))
assert _r["rows"] == 3 and _r["expected"] == 4 and _r["missing"] == 1 and _r["duplicates"] == 0

# the cancelling case: one gap AND one duplicate -> row count looks perfect but data isn't
_idx = pd.to_datetime(["2024-01-01 00:00", "2024-01-01 00:05", "2024-01-01 00:05", "2024-01-01 00:15"])
_r = check_timeline(pd.DataFrame({"a": [1, 2, 3, 4]}, index=_idx))
assert _r["rows"] == _r["expected"] == 4          # counts match...
assert _r["missing"] == 1 and _r["duplicates"] == 1   # ...but it's still broken
print("Task 1.2a tests passed")

Task 1.2a tests passed


In [414]:
def split_years(df:pd.DataFrame):

    """Split into (year1, year2) at exactly one year after the first timestamp.

    Uses DateOffset rather than a fixed 365 days so leap years split correctly.
    """
    
    cut = df.index.min() + pd.DateOffset(years=1)
    return df[df.index < cut], df[df.index >= cut]

In [415]:
"""Tests for Task 1.2b"""
_idx2 = pd.date_range("2023-01-01", "2024-12-31 23:55", freq="5min")
_y1, _y2 = split_years(pd.DataFrame({"a": 1}, index=_idx2))
assert _y1.index.max() < _y2.index.min()                      # no overlap
assert len(_y1) + len(_y2) == len(_idx2)                       # nothing lost
assert len(_y1) == 365 * 288 and len(_y2) == 366 * 288
print("Task 1.2b tests passed")

Task 1.2b tests passed


In [416]:
timeline = check_timeline(df)
print(pd.Series(timeline))
df1, df2 = split_years(df)
print("Year 1 rows:", len(df1), "| Year 2 rows:", len(df2))
pd.concat({"year1_nulls": df1.isna().sum(), "year2_nulls": df2.isna().sum()}, axis=1)

rows                       210240
expected                   210240
missing                         0
duplicates                      0
monotonic                    True
start         2022-01-01 00:00:00
end           2023-12-31 23:55:00
dtype: object
Year 1 rows: 105120 | Year 2 rows: 105120


,year1_nulls,year2_nulls
irradiance_W/m^2,184,866
temperature_C,184,866
solar_output_kW,184,105120
commercial_forecasted_solar_output_kW,105120,0
load_kW,184,866
spot_price_AUD/kWh,0,105120
tariff_import_flat_AUD/kWh,0,0
tariff_export_flat_AUD/kWh,0,0
tariff_import_tou_AUD/kWh,0,0
tariff_export_tou_AUD/kWh,0,0


In [417]:
sensor_cols = [COL["irr"], COL["temp"], COL["load"]]

# Are the sensor gaps the same rows? (True = yes)
blank = df[sensor_cols].isna()
print("Same rows across sensor columns:", (blank.all(axis=1) == blank.any(axis=1)).all())

# Group consecutive blank rows into gaps: start, end, length
is_gap = blank.any(axis=1)
gap_id = (is_gap != is_gap.shift()).cumsum()
gaps = (df[is_gap].groupby(gap_id[is_gap])
        .apply(lambda g: pd.Series({"start": g.index.min(), "end": g.index.max(), "intervals": len(g)})))
gaps["hours"] = gaps["intervals"] * DT_H
print(gaps.sort_values("intervals", ascending=False).head(15))
print("Number of gaps:", len(gaps))


Same rows across sensor columns: True
                 start                 end  intervals      hours
32 2023-07-01 22:20:00 2023-07-02 22:15:00        288  24.000000
12 2022-12-27 14:10:00 2022-12-27 19:50:00         69   5.750000
36 2023-08-12 07:10:00 2023-08-12 12:40:00         67   5.583333
30 2023-06-23 06:45:00 2023-06-23 12:15:00         67   5.583333
18 2023-03-15 13:50:00 2023-03-15 18:40:00         59   4.916667
42 2023-12-17 22:45:00 2023-12-18 03:25:00         57   4.750000
34 2023-07-23 22:15:00 2023-07-24 02:40:00         54   4.500000
28 2023-06-17 06:45:00 2023-06-17 10:45:00         49   4.083333
2  2022-03-13 10:50:00 2022-03-13 14:45:00         48   4.000000
38 2023-10-10 04:05:00 2023-10-10 08:00:00         48   4.000000
24 2023-04-29 07:10:00 2023-04-29 11:05:00         48   4.000000
22 2023-03-27 05:15:00 2023-03-27 07:10:00         24   2.000000
10 2022-09-13 15:55:00 2022-09-13 17:40:00         22   1.833333
44 2023-12-29 06:20:00 2023-12-29 08:05:00         2

> **Finding 1.2 - Completeness.** The timeline is complete: 210,240 of 210,240 expected 5-minute intervals, with no missing timestamps and no duplicates. Because there are no gaps or repeats around October and April, the timestamps are on a fixed offset with no daylight-saving jumps, consistent with NEM time (AEST); this is checked against solar noon in Part 4.
>
> Some columns are empty by design. Measured PV exists only in Year 1 and the commercial forecast only in Year 2, so the two are exact complements. There are no Year 2 spot prices, so actual 2023 NSW1 prices must be sourced from AEMO to settle the Year 2 wholesale bill.
>
> A complete timeline is not complete data. 22 data outages leave whole rows blank: the same rows in irradiance, temperature, load and PV (184 intervals in Year 1, 866 in Year 2, about 0.5% of the data). They range up to 24 hours (1–2 July 2023), and several cover the morning solar ramp. Handling is deferred to Part 2.

## Task 1.3 - Plausibility of each column

### Background: present is not the same as possible
Task 1.2 showed the data exists; this task checks it is physically possible. Each column has a hard physical bound: irradiance and load cannot be negative, PV is zero at night and capped by system size, and NEM spot prices sit between the market floor (-$1,000/MWh) and the market price cap. A value outside its bound means a unit or sensor problem that every later calculation would inherit. Year 1 and Year 2 are summarised separately so measured PV (Year 1) can be compared like-for-like with the commercial forecast (Year 2).

In [418]:
# Ranges per year (Year 2 PV is the forecast, so compare like with like)
desc = pd.concat({"Y1": df1.describe().T, "Y2": df2.describe().T})
desc[["min", "50%", "max", "mean"]].round(3)

min     50%      max     mean
Y1 irradiance_W/m^2                        0.00  13.000  1137.00  168.650
   temperature_C                          -8.20   6.400    24.30    6.816
   solar_output_kW                         0.00   0.100    10.31    1.263
   commercial_forecasted_solar_output_kW    NaN     NaN      NaN      NaN
   load_kW                                 0.17   0.570     5.87    1.212
   spot_price_AUD/kWh                     -1.00   0.134    15.50    0.183
   tariff_import_flat_AUD/kWh              0.30   0.300     0.30    0.300
   tariff_export_flat_AUD/kWh              0.05   0.050     0.05    0.050
   tariff_import_tou_AUD/kWh               0.15   0.250     0.40    0.242
   tariff_export_tou_AUD/kWh               0.05   0.050     0.05    0.050
Y2 irradiance_W/m^2                        0.00  16.000  1104.00  191.172
   temperature_C                         -10.10   7.400    26.20    7.805
   solar_output_kW                          NaN     NaN      NaN      NaN
   commercial_forecasted_solar_output_kW   0.00   0.120     8.25    1.458
   load_kW                                 0.17   0.570     5.87    1.215
   spot_price_AUD/kWh                       NaN     NaN      NaN      NaN
   tariff_import_flat_AUD/kWh              0.30   0.300     0.30    0.300
   tariff_export_flat_AUD/kWh              0.05   0.050     0.05    0.050
   tariff_import_tou_AUD/kWh               0.15   0.250     0.40    0.242
   tariff_export_tou_AUD/kWh               0.05   0.050     0.05    0.050

> **Finding 1.3 - Ranges.** Every column is physically plausible. Irradiance ranges from 0 to 1,137 W/m² (Year 1) and 0 to 1,104 W/m² (Year 2), consistent with clear-sky noon at about 1,000 W/m² plus brief cloud-edge enhancement and the site's high altitude. Temperature runs from −10.1 °C to 26.2 °C with a mean of about 7 °C, as expected for an alpine site, so cold-weather battery derating will matter (Part 5). Load is always positive (0.17–5.87 kW, mean 1.21 kW, about 29 kWh/day), so there is never zero standby load.
>
> The spot price reaches exactly −1.00 and 15.50 AUD/kWh, the NEM market floor (−1,000 AUD/MWh) and cap (15,500 AUD/MWh). This confirms the column is already in AUD/kWh and must not be divided by 1,000. Its mean of 183 AUD/MWh reflects the 2022 energy crisis.
>
> Measured PV peaks at 10.3 kW near 1,100 W/m², implying a system of roughly 10 kW (refined in Part 4). The Year 2 commercial forecast has a higher mean than Year 1 measured PV (1.46 vs 1.26 kW), in line with Year 2's sunnier irradiance, but a lower peak (8.25 vs 10.31 kW). Year 2 irradiance peaked only 3% lower than Year 1, so weather does not explain a 20% lower PV peak; this suggests the forecast is smoothed and misses short peaks, which is examined in Part 9.

## Task 1.4 - Tariff structure

### Background: read the tariff from the data, not from assumptions
The battery's value on a ToU tariff depends entirely on *when* each rate applies, so the windows must come from the data. `tou_schedule` builds an hour-by-hour table of the minimum and maximum rate, split by weekday and weekend. Taking min and max over the whole dataset tests three things at once: if they are equal for every hour, the boundaries fall on whole hours, the schedule does not change with season, and it applies identically to the days in each column.

In [419]:
def tou_schedule(rate: pd.Series) -> pd.DataFrame:
    """Hour-of-day x weekday/weekend table of the min and max rate in each hour.

    min == max in every cell means whole-hour boundaries and no seasonal change.
    min != max flags a mid-hour boundary or a rate that varies across the year.
    """
    t = pd.DataFrame({"hour": rate.index.hour,
                      "weekend": rate.index.dayofweek >= 5,
                      "rate": rate.values})
    return t.pivot_table(index="hour", columns="weekend", values="rate", aggfunc=["min", "max"])

In [420]:
"""Tests for Task 1.4"""
# Saturday 6 Jan and Monday 8 Jan 2024; peak 19:00-23:00, otherwise off-peak
_i = pd.date_range("2024-01-06", periods=288, freq="5min").append(pd.date_range("2024-01-08", periods=288, freq="5min"))
_r = pd.Series(np.where((_i.hour >= 19) & (_i.hour < 23), 0.40, 0.15), index=_i)
_s = tou_schedule(_r)
assert _s[("min", False)].loc[19] == 0.40 and _s[("min", True)].loc[19] == 0.40     # peak on both day types
assert _s[("max", False)].loc[3] == 0.15                                             # off-peak overnight
assert (_s["min"] == _s["max"]).all().all()                                          # whole-hour windows

# A boundary at 16:30 must show up as min != max in hour 16
_r2 = pd.Series(np.where(_i.hour + _i.minute / 60 >= 16.5, 0.40, 0.15), index=_i)
_s2 = tou_schedule(_r2)
assert _s2[("min", False)].loc[16] == 0.15 and _s2[("max", False)].loc[16] == 0.40
print("Task 1.4 tests passed")

Task 1.4 tests passed


In [421]:
for c in ["flat_imp", "flat_exp", "tou_imp", "tou_exp"]:
    print(f"{c:9s}", sorted(df[COL[c]].dropna().unique()))
tou_schedule(df[COL["tou_imp"]])

flat_imp  [np.float64(0.3)]
flat_exp  [np.float64(0.05)]
tou_imp   [np.float64(0.15), np.float64(0.25), np.float64(0.4)]
tou_exp   [np.float64(0.05)]


min         max      
weekend False True  False True 
hour                           
0        0.15  0.15  0.15  0.15
1        0.15  0.15  0.15  0.15
2        0.15  0.15  0.15  0.15
3        0.15  0.15  0.15  0.15
4        0.15  0.15  0.15  0.15
5        0.15  0.15  0.15  0.15
6        0.15  0.15  0.15  0.15
7        0.25  0.25  0.25  0.25
8        0.25  0.25  0.25  0.25
9        0.25  0.25  0.25  0.25
10       0.25  0.25  0.25  0.25
11       0.25  0.25  0.25  0.25
12       0.25  0.25  0.25  0.25
13       0.25  0.25  0.25  0.25
14       0.25  0.25  0.25  0.25
15       0.25  0.25  0.25  0.25
16       0.25  0.25  0.25  0.25
17       0.25  0.25  0.25  0.25
18       0.25  0.25  0.25  0.25
19       0.40  0.40  0.40  0.40
20       0.40  0.40  0.40  0.40
21       0.40  0.40  0.40  0.40
22       0.40  0.40  0.40  0.40
23       0.15  0.15  0.15  0.15

In [422]:
# What is one kWh moved by the battery worth on each tariff? (before losses)
h = df.index.hour
n_days = df.index.normalize().nunique()
peak_kwh_day = df.loc[(h >= 19) & (h < 23), COL["load"]].sum() * DT_H / n_days
print(f"Average load in the 19:00-23:00 peak window: {peak_kwh_day:.1f} kWh/day")

exp_rate = 0.05
pd.DataFrame({
    "store own solar, use later (AUD/kWh)":   {"Flat": 0.30 - exp_rate, "ToU into off-peak": 0.15 - exp_rate,
                                                "ToU into shoulder": 0.25 - exp_rate, "ToU into peak": 0.40 - exp_rate},
    "charge from grid off-peak, use later":  {"Flat": 0.0, "ToU into off-peak": 0.0,
                                                "ToU into shoulder": 0.25 - 0.15, "ToU into peak": 0.40 - 0.15},
}).round(2)

Average load in the 19:00-23:00 peak window: 2.6 kWh/day


,"store own solar, use later (AUD/kWh)","charge from grid off-peak, use later"
Flat,0.25,0.00
ToU into off-peak,0.10,0.00
ToU into shoulder,0.20,0.10
ToU into peak,0.35,0.25


> **Finding 1.4 - Tariffs.** The tariff columns match the supplied metadata (flat 0.30/0.05 AUD/kWh; ToU 0.15/0.25/0.40 AUD/kWh import, 0.05 export). The metadata gives the rates but not their timing; the data shows:
>
> | Window | Hours | Import (AUD/kWh) |
> |---|---|---|
> | Off-peak | 23:00 - 07:00| 0.15 |
> | Shoulder | 07:00 - 19:00 | 0.25 |
> | Peak | 19:00 - 23:00 | 0.40 |
>
> The schedule is identical on weekdays and weekends with no seasonal change. Export earns 0.05 AUD/kWh under every tariff.
>
> **So what for the battery.** Export is worth less than every import rate, so on every tariff a stored kWh of solar beats an exported one. On the **flat** tariff that is the battery's only value (0.25 AUD/kWh shifted, before losses); there is no price spread to arbitrage. On **ToU** the value depends on *when* the stored energy is used: 0.35 AUD/kWh into the peak, 0.20 into the shoulder, but only 0.10 overnight. ToU also opens **grid arbitrage**: charging off-peak and discharging in the peak is worth 0.25 AUD/kWh before losses, which matters in winter when there is little surplus solar. The peak window is only 4 hours and carries about 2.6 kWh/day of load, so a battery sized for the peak alone would be small; the sizing question (Part 4) is how much of the evening and overnight load is worth covering.


## Task 1.5 - Is the load independent across years?

### Background: check whether two years are really two years
The Year 1 and Year 2 load summaries in Task 1.3 were almost identical. `compare_years` lines the two years up interval by interval (by calendar position) and measures how often the values are identical, how correlated they are, and how many gaps in each year could be filled from the other. Irradiance is run through the same check as a control: real weather should not repeat.

In [423]:
def compare_years(y1: pd.Series, y2: pd.Series) -> dict:
    """Compare two equal-length years aligned by calendar position.

    Positional alignment needs equal-length years (true for 2022/2023, both 365 days);
    a leap year would shift every interval after 29 Feb, so the function refuses.
    """
    a, b = np.asarray(y1, float), np.asarray(y2, float)
    if len(a) != len(b):
        raise ValueError(f"Years differ in length ({len(a)} vs {len(b)}): positional alignment invalid")
    both = ~np.isnan(a) & ~np.isnan(b)
    return dict(
        identical_frac=float(np.isclose(a[both], b[both]).mean()),
        corr=float(np.corrcoef(a[both], b[both])[0, 1]),
        y1_gaps=int(np.isnan(a).sum()), y1_fillable_from_y2=int((np.isnan(a) & ~np.isnan(b)).sum()),
        y2_gaps=int(np.isnan(b).sum()), y2_fillable_from_y1=int((np.isnan(b) & ~np.isnan(a)).sum()),
        missing_both=int((np.isnan(a) & np.isnan(b)).sum()),
    )

In [424]:
"""Tests for Task 1.5"""
_c = compare_years([1, 2, np.nan, 4], [1, np.nan, 3, 4])
assert _c["identical_frac"] == 1.0                                   # compared only where both exist
assert _c["y1_fillable_from_y2"] == 1 and _c["y2_fillable_from_y1"] == 1 and _c["missing_both"] == 0

_c = compare_years([1, 2, 3, 4], [1, 2, 3, 9])
assert _c["identical_frac"] == 0.75

try:
    compare_years([1, 2, 3], [1, 2])
    raise AssertionError("should have refused unequal lengths")
except ValueError:
    pass
print("Task 1.5 tests passed")

Task 1.5 tests passed


In [425]:
pd.DataFrame({
    "load": compare_years(df1[COL["load"]], df2[COL["load"]]),
    "irradiance (control)": compare_years(df1[COL["irr"]], df2[COL["irr"]]),
}).round(4)

,load,irradiance (control)
identical_frac,1.0,0.4570
corr,1.0,0.8482
y1_gaps,184.0,184.0000
y1_fillable_from_y2,184.0,184.0000
y2_gaps,866.0,866.0000
y2_fillable_from_y1,866.0,866.0000
missing_both,0.0,0.0000


In [426]:
# Weekday labels: 1 Jan 2022 was a Saturday, 1 Jan 2023 a Sunday, so a copied year shifts weekdays by one day
print(df1.index[0].day_name(), "|", df2.index[0].day_name())
(df1[COL["load"]].groupby(df1.index.dayofweek).mean() * 24).round(1).rename("Y1 load kWh/day by weekday (0 = Mon)")

Saturday | Sunday


timestamp
0    28.8
1    28.8
2    28.2
3    29.1
4    29.6
5    29.8
6    29.3
Name: Y1 load kWh/day by weekday (0 = Mon), dtype: float64

> **Finding 1.5 - Load is repeated.** Year 2 load equals Year 1 load at every interval where both exist (identical fraction and correlation both 1.0), while irradiance does not (46% identical, r = 0.85). So this is a property of the load data, not a bug in the check. Implications:
> 1. Year 2 is not independent load data. Any difference in battery value between the years comes only from PV, weather and prices, which makes the years a clean comparison of those drivers.
> 2. The copy is by calendar date, so Year 2 weekday labels are shifted one day against the load behaviour. Year 1 load barely varies by weekday (28.2-29.8 kWh/day), so this has no practical effect, but day-of-week should not be used as a feature.
> 3. No interval is missing in both years, so all 1,050 load gaps can be filled exactly from the other year (Part 2).

## Task 1.6 - Spurious data

### Background: in range is not the same as consistent
Task 1.3 checked that every value is *possible* on its own. A value can still be wrong if it contradicts how the sensor behaves over time, or what another sensor says at the same moment. `quality_report` runs three kinds of check:

| Kind | Question | Checks |
|---|---|---|
| Range | Is each value possible on its own? | Negative or zero load, negative irradiance or PV, prices outside the NEM floor and cap. These repeat Task 1.3 as a formal record |
| Behaviour over time | Does the sensor behave like a real sensor? | Longest constant run (stuck meter or sensor), largest 5-minute temperature jump |
| Cross-sensor | Do two sensors measuring the same sunlight agree? | PV when irradiance is zero, no PV under strong sun (outage, snow or inverter trip, plausible at an alpine site), PV above system capacity, daily PV-to-irradiance ratio |

Cross-sensor checks are the strongest evidence, because one sensor checks the other. Each year is checked against its own PV column: measured PV in Year 1, the commercial forecast in Year 2. `quality_report` is adapted from my Assignment 1 data-quality check, with the PV column and system capacity as arguments.

In [427]:
def longest_constant_run(s: pd.Series) -> int:
    """Length (in intervals) of the longest run of identical consecutive values."""
    s = s.dropna()
    return int(s.groupby((s != s.shift()).cumsum()).size().max()) if len(s) else 0


def quality_report(df: pd.DataFrame, pv_col: str, cap_kw: float, col: dict = COL) -> pd.DataFrame:
    """Physical-plausibility checks in one table. Columns: check, value, note.

    pv_col: which PV column to check (measured in Year 1, commercial forecast in Year 2).
    cap_kw: estimated PV system size, used for the above-capacity check.
    """
    irr, load = df[col["irr"]], df[col["load"]]
    rows = [
        ("Negative load intervals", int((load < 0).sum()), "Should be 0 (no export-side meter in load)"),
        ("Zero-load intervals", int((load == 0).sum()), "A real home always has standby load"),
        ("Longest constant load run (intervals)", longest_constant_run(load), "Stuck meter if very long"),
        ("Longest constant temperature run (intervals)", longest_constant_run(df[col["temp"]]), "Stuck sensor if very long"),
        ("Max 5-min temperature jump (degC)", float(df[col["temp"]].diff().abs().max()), "Implausible if large"),
        ("Negative irradiance intervals", int((irr < 0).sum()), "Should be 0"),
    ]
    s = df[pv_col].dropna()
    if len(s):
        i = irr.reindex(s.index)
        rows += [
            ("PV rows checked", len(s), "Proves the check actually ran"),
            ("Solar > 0.01 kW at zero irradiance", int(((i == 0) & (s > 0.01)).sum()), "Should be 0"),
            ("Negative solar intervals", int((s < 0).sum()), "Should be 0"),
            ("Daytime (irr>100) intervals with solar <= 0.01 kW",
             int(((i > 100) & (s <= 0.01)).sum()), "Outage, snow or fault"),
            (f"Solar above 1.1 x {cap_kw} kW", int((s > 1.1 * cap_kw).sum()), "Above system capacity"),
        ]
    p = df[col["rrp"]].dropna()
    if len(p):
        rows += [("Prices outside [-1.00, 15.50] $/kWh", int(((p < -1.00) | (p > 15.50)).sum()),
                  "Outside 2022 NEM floor/cap: check units"),
                 ("Negative-price intervals", int((p < 0).sum()), "Drives the curtailment logic")]
    return pd.DataFrame(rows, columns=["check", "value", "note"])

In [428]:
"""Tests for Task 1.6"""
assert longest_constant_run(pd.Series([1, 1, 1, 2, 2])) == 3
assert longest_constant_run(pd.Series([np.nan, np.nan])) == 0            # empty column must not crash

_idx = pd.date_range("2022-01-01 12:00", periods=4, freq="5min")
_q = pd.DataFrame({
    COL["irr"]:      [0, 0, 500, 500],
    COL["load"]:     [0.0, 0.3, 0.3, 0.3],
    COL["temp"]:     [5, 5, 5, 6],
    COL["solar"]:    [0.0, 0.5, 0.0, 12.0],        # PV in the dark, no PV in sun, above 1.1 x 10 kW
    COL["solar_fc"]: np.nan,
    COL["rrp"]:      [0.10, -0.05, 16.0, 0.10],    # one negative price, one above the cap
}, index=_idx)
_r = quality_report(_q, COL["solar"], cap_kw=10).set_index("check")["value"]
assert _r["Zero-load intervals"] == 1
assert _r["Solar > 0.01 kW at zero irradiance"] == 1
assert _r["Daytime (irr>100) intervals with solar <= 0.01 kW"] == 1
assert _r["Solar above 1.1 x 10 kW"] == 1
assert _r["Prices outside [-1.00, 15.50] $/kWh"] == 1 and _r["Negative-price intervals"] == 1

# Year 2 shape: measured PV empty, forecast present -> the forecast is checked, not silently skipped
_y2 = _q.assign(**{COL["solar"]: np.nan, COL["solar_fc"]: [0.0, 0.5, 0.0, 12.0]})
_r2 = quality_report(_y2, COL["solar_fc"], cap_kw=10).set_index("check")["value"]
assert _r2["PV rows checked"] == 4 and _r2["Solar above 1.1 x 10 kW"] == 1
# an empty PV column skips the PV checks instead of reporting false zeros
assert "PV rows checked" not in quality_report(_y2, COL["solar"], cap_kw=10)["check"].values
print("Task 1.6 tests passed")

Task 1.6 tests passed


In [429]:
pd.concat({"Y1": quality_report(df1, COL["solar"], 10).set_index("check")["value"],
           "Y2": quality_report(df2, COL["solar_fc"], 10).set_index("check")["value"]}, axis=1)

,Y1,Y2
check,,
Negative load intervals,0.0,0.0
Zero-load intervals,0.0,0.0
Longest constant load run (intervals),14.0,14.0
Longest constant temperature run (intervals),73.0,81.0
Max 5-min temperature jump (degC),0.8,0.5
Negative irradiance intervals,0.0,0.0
PV rows checked,104936.0,105120.0
Solar > 0.01 kW at zero irradiance,0.0,0.0
Negative solar intervals,0.0,0.0


In [430]:
def daily_pv_irr_ratio(pv: pd.Series, irr: pd.Series) -> pd.Series:
    """Daily PV energy divided by daily irradiation (kWh per kWh/m^2, i.e. the array's effective kW).

    Uses complete days only: an irradiance outage with no matching PV gap divides
    by too little and inflates the ratio (to infinity if only night-time irradiance is left).
    """
    complete = pd.concat([pv, irr], axis=1).notna().all(axis=1).groupby(pv.index.normalize()).all()
    irr_day = irr.resample("D").sum() / 1000
    ratio = pv.resample("D").sum() / irr_day
    return ratio[complete & (irr_day > 0)]

In [431]:
"""Tests for Task 1.6 (ratio)"""
_i = pd.date_range("2024-01-01", periods=2 * 288, freq="5min")
_irr = pd.Series(np.where((_i.hour >= 6) & (_i.hour < 18), 800.0, 0.0), index=_i)
_pv = _irr / 1000 * 7.5                                     # a 7.5 kW-effective array
_irr_gap = _irr.copy(); _irr_gap.loc["2024-01-02 08:00":"2024-01-02 12:00"] = np.nan
_r = daily_pv_irr_ratio(_pv, _irr_gap)
assert np.isclose(_r.loc["2024-01-01"], 7.5)                # complete day: exact
assert pd.Timestamp("2024-01-02") not in _r.index           # day with a gap: excluded, not inflated
print("Task 1.6 ratio tests passed")

Task 1.6 ratio tests passed


In [432]:
ratio = pd.DataFrame({
    "Y1 measured":   daily_pv_irr_ratio(df1[COL["solar"]],    df1[COL["irr"]]),
    "Y2 commercial": daily_pv_irr_ratio(df2[COL["solar_fc"]], df2[COL["irr"]]),
})
summary = ratio.describe().loc[["count", "min", "50%", "max"]].round(2)
summary.loc["min day"] = ratio.idxmin().dt.date
summary.loc["max day"] = ratio.idxmax().dt.date
summary

,Y1 measured,Y2 commercial
count,359.0,346.0
min,6.95,7.29
50%,7.57,7.72
max,8.09,8.12
min day,2022-01-23,2023-12-12
max day,2022-06-29,2023-06-20


In [433]:
def longest_run_details(s: pd.Series) -> dict:
    """Where the longest constant run is: start, end, length and value (to judge stuck sensor vs calm night)."""
    s = s.dropna()
    groups = (s != s.shift()).cumsum()
    run = s[groups == s.groupby(groups).size().idxmax()]
    return dict(intervals=len(run), hours=round(len(run) * DT_H, 2),
                start=run.index[0], end=run.index[-1], value_C=run.iloc[0])

pd.DataFrame({"Y1": longest_run_details(df1[COL["temp"]]),
              "Y2": longest_run_details(df2[COL["temp"]])})

,Y1,Y2
intervals,73,81
hours,6.08,6.75
start,2022-08-24 18:30:00,2023-07-05 02:30:00
end,2022-08-25 00:30:00,2023-07-05 09:10:00
value_C,-0.8,4.2


In [434]:
# Is 5-minute irradiance real 5-minute data? If it was interpolated from hourly values,
# the 5-minute steps inside each hour are all the same size (+/-1 W/m^2 for rounding).
irr1 = df1[COL["irr"]]
step = irr1.diff()
inside_hour = (irr1.index.minute != 0) & step.notna()
spread = step[inside_hour].groupby(irr1.index[inside_hour].floor("h")).agg(lambda x: x.max() - x.min())
daylight = irr1.resample("h").mean().reindex(spread.index) > 50
print(f"Daylight hours whose 5-min irradiance steps are all equal: {(spread[daylight] <= 1).mean():.1%}")

Daylight hours whose 5-min irradiance steps are all equal: 100.0%


> **Finding 1.6 - No spurious data.** Every check passes in both years (104,936 measured PV rows in Year 1 and 105,120 commercial-forecast rows in Year 2): no PV in the dark, no PV dropout under strong sun, nothing above the ~10 kW system capacity, no zero or negative load, and every price within the NEM floor and cap. The 3,114 negative-price intervals in Year 1 (about 3% of the year) are real market events, not errors; they are when PV can be curtailed (Part 7).
>
> **Temperature.** The longest constant runs (73 intervals, about 6 h, in Year 1; 81, about 6.75 h, in Year 2) are both overnight in winter: from the evening of 24 Aug 2022 at -0.8 °C, and the early morning of 5 Jul 2023 at 4.2 °C. Temperature is recorded to 0.1 °C and never moves more than 0.8 °C in 5 minutes, so a flat reading through a calm alpine night is expected. This is not a stuck sensor, so the data is kept.
>
> **PV vs irradiance.** On complete days the daily PV-to-irradiance ratio is very stable: 6.95 to 8.09 (median 7.57) in Year 1 and 7.29 to 8.12 (median 7.72) in Year 2. There are no snow-covered, soiled or faulted days to remove. The ratio is lowest in summer and highest in winter, as expected from the negative temperature coefficient (cold panels are more efficient). It must be computed on complete days only: on days with an irradiance outage it is inflated, because the PV columns have no matching gap. The ratio (~7.6 kW per kW/m²) is also a first estimate of effective array size for Part 4.
>
> **Irradiance resolution.** In every daylight hour, the 5-minute irradiance rises in equal steps, so it is **hourly data interpolated to 5 minutes**, while measured PV is true 5-minute data. Cloud transients therefore appear in PV but not in irradiance. A weather-based PV model (Part 9) cannot predict sub-hourly dips, which sets a floor under its 5-minute error.

## Part 1 summary - what we are working with

Part 1 diagnosed the data without changing it. Part 2 fixes only what this list says needs fixing.

| # | Issue | Found in | What happens next |
|---|---|---|---|
| 1 | 22 sensor outages blank irradiance, temperature, load and measured PV together (184 rows in Y1, 866 in Y2; up to 24 h) | 1.2 | Part 2: fill or flag, column by column |
| 2 | Measured PV only in Y1, commercial forecast only in Y2 | 1.2 | By design. Y2 "actual" PV = commercial forecast (per the brief) |
| 3 | No Y2 spot prices | 1.2 | Part 3: source 2023 NSW1 prices from AEMO |
| 4 | Y2 load is an exact copy of Y1, with weekdays shifted one day | 1.5 | Part 2: fill load gaps from the other year; no day-of-week features |
| 5 | Prices already in $/kWh; hit the floor and cap; 3,114 negative intervals | 1.3, 1.6 | Keep as-is; curtail PV at negative prices (Part 7) |
| 6 | Irradiance is hourly data interpolated to 5 min | 1.6 | Caveat for the PV model (Part 9) |
| 7 | No spurious data; temperature runs are calm winter nights | 1.6 | Nothing to remove |
| 8 | No daylight-saving jumps (likely AEST) | 1.2 | Confirm with solar noon (Part 4) |

# Part 2 - Data Cleaning

Part 1 diagnosed the data; this Part fixes only what the Part 1 summary lists. Filling a gap invents data, so each column is filled only if a later step needs it complete, using the simplest method that is defensible for that use.

## Task 2.1 - Cleaning decisions

| Column | Used later for | Gaps (Part 1) | Decision | Why |
|---|---|---|---|---|
| Load | every bill and optimisation, both years | 184 (Y1), 866 (Y2) | **Fill exactly** from the other year (Task 2.2) | The years are copies (Finding 1.5) and no interval is missing in both, so this recovers the true value |
| "Actual" PV in Y2 (= commercial forecast) | perfect-foresight optimisation and settlement | 0 | Nothing to fill | Complete by design |
| Measured PV (Y1) | training the PV model (Part 9) | 184 | **Do not fill**; drop those rows from training | Filling a training target would teach the model invented data |
| Irradiance, temperature | PV model inputs; battery temperature limits | 184 (Y1), 866 (Y2) | **Fill and flag** (Task 2.3) | Inputs must be complete for a forecast at every hour; flagged rows are left out when forecasts are scored |
| Spot price (Y1) | price forecast training | 0 | Nothing to fill | Y2 prices are sourced in Part 3 |
| Everything else | | | Keep as-is | No spurious data (Finding 1.6) |

## Task 2.2 - Fill load from the other year

In [435]:
def fill_from_other_year(s: pd.Series, offset=pd.DateOffset(years=1)):
    """Fill gaps with the value at the same calendar position one year earlier or later.

    Only valid because Finding 1.5 showed the two years of load are copies.
    Returns (filled series, boolean mask of rows that were filled).
    """
    prev = s.reindex(s.index - offset).set_axis(s.index)
    nxt = s.reindex(s.index + offset).set_axis(s.index)

    filled = s.fillna(prev).fillna(nxt)

    return filled, s.isna() & filled.notna()

In [436]:
"""Tests for Task 2.2"""
_i = pd.date_range("2022-01-01", periods=3, freq="D").append(pd.date_range("2023-01-01", periods=3, freq="D"))
_f, _m = fill_from_other_year(pd.Series([1, np.nan, 3, 1, 2, np.nan], index=_i))
assert _f.tolist() == [1, 2, 3, 1, 2, 3]                                  # filled in both directions
assert _m.tolist() == [False, True, False, False, False, True]            # only the gaps are flagged

# missing in BOTH years must stay missing, not be invented
_i2 = pd.to_datetime(["2022-01-01", "2022-01-02", "2023-01-01", "2023-01-02"])
_f2, _ = fill_from_other_year(pd.Series([np.nan, 1, np.nan, 1], index=_i2))
assert np.isnan(_f2.iloc[0]) and np.isnan(_f2.iloc[2])
print("Task 2.2 tests passed")

Task 2.2 tests passed


In [437]:
load_clean, load_filled = fill_from_other_year(df[COL["load"]])
print("Load gaps filled:", int(load_filled.sum()), "| still missing:", int(load_clean.isna().sum()))

Load gaps filled: 1050 | still missing: 0


## Task 2.3 - Fill irradiance and temperature

### Background: why not straight-line interpolation?
Weather is not copied between years (the irradiance control in Task 1.5), so it cannot be filled exactly. Straight-line interpolation is fine for a few minutes but wrong for these outages: the shortest is 1 hour and the longest is 24 hours (1-2 July 2023), and a line across a day erases the daily solar shape. Instead each missing value is the **mean of the same time of day on the 3 days either side**, which keeps night at zero and noon bright. The method is then tested on real data by blanking complete days and refilling them.

In [438]:
def fill_same_time_nearby_days(s: pd.Series, days: int = 3):
    """Fill each gap with the mean of the same time of day on the `days` days either side.

    Neighbouring days that are themselves missing are skipped by the mean.
    Returns (filled series, boolean mask of rows that were filled).
    """
    surrounding = [s.shift(freq=pd.Timedelta(days=k)).reindex(s.index) 
                   for k in range (-days,days+1) if k!=0]
    
    filled = s.fillna(pd.concat(surrounding, axis=1).mean(axis=1))

    return filled, s.isna() & filled.notna()



In [439]:
"""Tests for Task 2.3"""
_i = pd.date_range("2024-01-01", periods=5 * 24, freq="h")
_true = pd.Series(np.tile(np.r_[np.zeros(6), np.full(12, 500.0), np.zeros(6)], 5), index=_i)
_gap = _true.copy(); _gap.loc["2024-01-03"] = np.nan                 # one whole day missing
_f, _m = fill_same_time_nearby_days(_gap, days=2)
assert np.allclose(_f.values, _true.values)                          # daily shape restored, night stays zero
assert _m.sum() == 24
assert not np.allclose(_gap.interpolate().values, _true.values)      # a straight line would lose the noon peak
print("Task 2.3 tests passed")

Task 2.3 tests passed


In [440]:

rng = np.random.default_rng(RNG_SEED)
irr = df[COL["irr"]]
complete_days = irr.notna().groupby(irr.index.normalize()).all()
test_days = rng.choice(complete_days[complete_days].index[3:-3], size=50, replace=False)
blanked = irr.copy()
for d in test_days:
    blanked.loc[str(pd.Timestamp(d).date())] = np.nan
daylight = blanked.isna() & irr.notna() & (irr > 0)
err = pd.DataFrame({
    "nearby-days mean": (fill_same_time_nearby_days(blanked)[0] - irr)[daylight].abs(),
    "straight line":    (blanked.interpolate(method="time") - irr)[daylight].abs(),
})
print("Daylight MAE when refilling 50 whole blanked days (W/m^2):")
err.mean().round(1)

Daylight MAE when refilling 50 whole blanked days (W/m^2):


nearby-days mean     80.5
straight line       332.5
dtype: float64

In [441]:
weather, flags = {}, pd.DataFrame(index=df.index)
for c in ["irr", "temp"]:
    weather[c], flags[f"{c}_filled"] = fill_same_time_nearby_days(df[COL[c]])
    print(f"{c:5s} filled {int(flags[f'{c}_filled'].sum()):4d} | still missing {int(weather[c].isna().sum())}")
flags["load_filled"] = load_filled
flags["pv_meas_missing"] = df[COL["solar"]].isna() & (df.index < df2.index.min())   # Y1 rows to drop from PV training

irr   filled 1050 | still missing 0
temp  filled 1050 | still missing 0


In [442]:
# Figure 2.1 - Before/after: the 24-hour outage (1-2 July 2023), refilled two ways
win = slice("2023-06-30", "2023-07-03")
raw = df[COL["irr"]].loc[win]
gap = raw.isna()
straight = df[COL["irr"]].interpolate(method="time").loc[win]

fig = go.Figure()
fig.add_trace(go.Scatter(x=raw.index, y=raw, name="Measured",
                         line=dict(color="black", width=1.5)))
fig.add_trace(go.Scatter(x=raw.index, y=weather["irr"].loc[win].where(gap), name="Filled: same time on nearby days (used)",
                         line=dict(color="#1f77b4", width=2.5)))
fig.add_trace(go.Scatter(x=raw.index, y=straight.where(gap), name="Straight-line interpolation (rejected)",
                         line=dict(color="#d62728", width=2, dash="dash")))
fig.add_vrect(x0=raw.index[gap].min(), x1=raw.index[gap].max(), fillcolor="grey", opacity=0.15, line_width=0,
              annotation_text="24 h sensor outage", annotation_position="top left")
fig.update_layout(
    title="Filling the longest outage: the nearby-days fill keeps the daily solar shape; a straight line erases the day",
    xaxis_title="Time", yaxis_title="Irradiance (W/m²)", template="plotly_white", height=420,
    legend=dict(orientation="h", yanchor="top", y=-0.18, x=0))
fig.show()

*Figure 2.1 - The 24-hour outage of 1-2 July 2023. A straight line joins midnight to midnight (both zero), so it fills the whole day as darkness (peak 0 W/m²). The nearby-days fill gives a peak of about 280 W/m², in line with the measured days either side (180-490 W/m²).*

> **Finding 2.2-2.3 - Gaps.** All 1,050 load gaps were filled exactly from the other year. All 1,050 irradiance and temperature gaps were filled from the same time of day on the 3 days either side. Tested on 50 real days blanked on purpose, this gives a daylight error of about 80 W/m², against 333 W/m² for straight-line interpolation. The error is not small, because the weather on a given day cannot be known. That is acceptable: filled weather rows are only inputs (to the Year 2 PV forecast and the battery temperature limits), cover under 1% of each year, and are flagged so no forecast is scored on them. Measured Year 1 PV was not filled, so the PV model never trains on invented targets.

## Task 2.4 - Assemble the clean dataset
One analysis-ready frame `dfc` with short column names, the brief's definition of "actual" PV (measured in Year 1, commercial forecast in Year 2), and the fill flags. **Everything from Part 3 onwards uses `dfc`, never the raw `df`.** The contract checks below prove the frame matches the decisions in Task 2.1.

In [443]:
def build_clean(df: pd.DataFrame, load: pd.Series, weather: dict, flags: pd.DataFrame, cut) -> pd.DataFrame:
    """Analysis-ready frame. 'pv_kW' follows the brief: measured PV in Year 1, commercial forecast in Year 2.

    Columns: year, load_kW, pv_kW, pv_meas_kW, pv_comm_kW, irr_Wm2, temp_C, rrp_kWh,
    flat_imp, flat_exp, tou_imp, tou_exp, then the flag columns joined on.
    """
    y1 = df.index < cut
    out = pd.DataFrame({
        "year":         np.where(y1,1,2),
        "load_kW":      load,
        "pv_meas_kW":   df[COL["solar"]],
        "pv_comm_kW":   df[COL["solar_fc"]],
        "pv_kW":        df[COL["solar"]].where(y1, df[COL["solar_fc"]]),
        "irr_Wm2":      weather["irr"],
        "temp_C":       weather["temp"],
        "rrp_kWh":      df[COL["rrp"]],

        **{c: df[COL[c]] for c in ["flat_imp", "flat_exp", "tou_imp", "tou_exp"]},

    }, index =df.index)

    return out.join(flags)

In [444]:
dfc = build_clean(df, load_clean, weather, flags, cut=df2.index.min())

# Contract checks: what every later Part is allowed to assume (matches Task 2.1)
must_be_complete = ["load_kW", "irr_Wm2", "temp_C", "flat_imp", "flat_exp", "tou_imp", "tou_exp"]
assert dfc[must_be_complete].notna().all().all(),                         "gaps left in a column that must be complete"
assert dfc.loc[dfc.year == 2, "pv_kW"].notna().all(),                      "Year 2 'actual' PV must be complete"
assert dfc["pv_kW"].isna().sum() == dfc["pv_meas_missing"].sum(),          "only the unfilled Y1 PV rows may be blank"
assert dfc.loc[dfc.year == 2, "pv_kW"].equals(dfc.loc[dfc.year == 2, "pv_comm_kW"])
assert dfc.loc[dfc.year == 2, "rrp_kWh"].isna().all()                      # Y2 prices still to be sourced (Part 3)
print("Clean dataset:", dfc.shape, "| all contract checks passed")
dfc.isna().sum()[lambda s: s > 0]

Clean dataset: (210240, 16) | all contract checks passed


pv_meas_kW    105304
pv_comm_kW    105120
pv_kW            184
rrp_kWh       105120
dtype: int64

> **Finding 2.4 - Clean dataset.** `dfc` holds 210,240 rows and 16 columns. Load, weather and tariffs are complete; "actual" PV is complete in Year 2 and blank only for the 184 Year 1 rows deliberately left unfilled; Year 2 spot prices remain blank until Part 3. Every filled value is flagged (`load_filled`, `irr_filled`, `temp_filled`), so any later result can be checked with and without filled data.

# Part 3 - Tariffs and Prices

Part 3 builds the price every later Part uses: the import and export price for each of the three tariffs at every 5-minute interval. Flat and ToU come straight from the site data (Task 1.4). Wholesale pass-through needs the spot price for both years. The site data only has Year 1 prices, so Year 2 is sourced from AEMO first.

## Task 3.1 - Source the 2023 spot prices from AEMO

### Background: download once, align carefully
`load_spot_prices` is adapted from my thesis downloader. It is cached twice, so a full top-to-bottom run never downloads again: one raw CSV per month, and one combined CSV for the whole range, both in `PRICE_DIR`. Two changes from the thesis version: a month that fails to download now **raises an error** instead of being skipped, because a silently missing month would corrupt the annual bill; and it returns $/MWh exactly as published, with conversion to $/kWh done in one visible place.

AEMO stamps each 5-minute interval at its **end** (`SETTLEMENTDATE` 00:05 covers 00:00-00:05). The site data could use either convention, so rather than assume, January 2022 is downloaded as well and compared against the Year 1 prices already in the site data. The convention that matches exactly is the one used for 2023.

In [445]:
def months_between(start: str, end: str) -> list:
    """'YYYYMM' strings for every month from start to end inclusive, e.g. ('2022-11', '2023-02')."""
    return [p.strftime("%Y%m") for p in pd.period_range(start, end, freq="M")]


def load_spot_prices(region: str, start: str, end: str, cache_dir) -> pd.Series:
    """AEMO 5-minute regional reference price ($/MWh) for whole months `start`..`end`.

    Index is AEMO's SETTLEMENTDATE (interval END, NEM time). Cached in `cache_dir`:
      1. a combined CSV for this region and range: read immediately if it exists;
      2. one raw CSV per month: only missing months are downloaded.
    """
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    combined = cache_dir / f"spot_{region}_{start}_{end}.csv"
    if combined.exists():
        return pd.read_csv(combined, index_col=0, parse_dates=True)["rrp_MWh"]

    frames = []
    for ym in months_between(start, end):
        month_file = cache_dir / f"PRICE_AND_DEMAND_{ym}_{region}.csv"
        if not month_file.exists():
            url = AEMO_PRICE_URL.format(ym=ym, region=region)
            print(f"Downloading {url}")
            resp = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
            resp.raise_for_status()                      # fail loudly: a missing month corrupts the year
            if not resp.content.lstrip().startswith(b"REGION"):  # AEMO can return an HTML page with status 200
                raise ValueError(f"{url} did not return a price CSV; download it manually into {cache_dir}")
            month_file.write_bytes(resp.content)         # only a validated file is cached
        frames.append(pd.read_csv(month_file))

    raw = pd.concat(frames, ignore_index=True)
    s = (raw.assign(settlement_end=pd.to_datetime(raw["SETTLEMENTDATE"]))
            .set_index("settlement_end")["RRP"].sort_index().rename("rrp_MWh"))
    s = s[~s.index.duplicated()]
    s.to_csv(combined)
    return s


def to_period_start(s: pd.Series, minutes: int = INTERVAL_MIN) -> pd.Series:
    """Re-stamp interval-END timestamps (AEMO) as interval-START timestamps (site data)."""
    out = s.copy()
    out.index = out.index - pd.Timedelta(minutes=minutes)
    out.index.name = "timestamp"
    return out

In [446]:
"""Tests for Task 3.1 (offline: uses a fake cached month, never the network)"""
import tempfile
assert months_between("2022-11", "2023-02") == ["202211", "202212", "202301", "202302"]

with tempfile.TemporaryDirectory() as _d:
    _d = Path(_d)
    pd.DataFrame({"REGION": "NSW1", "SETTLEMENTDATE": ["2022/01/01 00:05:00", "2022/01/01 00:10:00"],
                  "TOTALDEMAND": 1.0, "RRP": [100.0, -50.0], "PERIODTYPE": "TRADE"}
                ).to_csv(_d / "PRICE_AND_DEMAND_202201_NSW1.csv", index=False)
    _s = load_spot_prices("NSW1", "2022-01", "2022-01", _d)            # reads the cached month, no download
    assert _s.tolist() == [100.0, -50.0]
    assert (_d / "spot_NSW1_2022-01_2022-01.csv").exists()              # combined cache written
    assert load_spot_prices("NSW1", "2022-01", "2022-01", _d).equals(_s)  # second call: combined cache, same data

assert to_period_start(_s).index[0] == pd.Timestamp("2022-01-01 00:00")  # interval ending 00:05 starts at 00:00
print("Task 3.1 tests passed")

Task 3.1 tests passed


In [447]:
# Alignment check: which timestamp convention reproduces the Year 1 prices in the site data?
jan22 = load_spot_prices("NSW1", "2022-01", "2022-01", PRICE_DIR) / 1000          # $/MWh -> $/kWh
site_jan22 = df.loc["2022-01", COL["rrp"]]
conventions = {"AEMO stamp = interval END (as published)": jan22,
               "AEMO stamp shifted to interval START":    to_period_start(jan22)}
pd.DataFrame({
    name: {"intervals compared": int(site_jan22.index.isin(s.index).sum()),
           "exact matches (to $0.00001/kWh)": f"{np.isclose(site_jan22, s.reindex(site_jan22.index), atol=1e-5).mean():.1%}",
           "max abs difference ($/kWh)": float((site_jan22 - s.reindex(site_jan22.index)).abs().max())}
    for name, s in conventions.items()
}).T

,intervals compared,exact matches (to $0.00001/kWh),max abs difference ($/kWh)
AEMO stamp = interval END (as published),8927,100.0%,0.0
AEMO stamp shifted to interval START,8928,22.7%,0.37464


In [448]:
# Year 2 spot price ($/kWh). The alignment check showed the site data uses AEMO's interval-END stamps
# as published, so no shift. The first Year 2 interval (2023-01-01 00:00) ends the last AEMO interval
# of December 2022, so the download starts in December 2022.
spot_y2 = load_spot_prices("NSW1", "2022-12", "2023-12", PRICE_DIR) / 1000
spot_y2 = spot_y2.reindex(dfc.index[dfc.year == 2])
assert spot_y2.notna().all(), f"{spot_y2.isna().sum()} Year 2 intervals have no AEMO price"

dfc.loc[dfc.year == 2, "rrp_kWh"] = spot_y2.values
assert dfc["rrp_kWh"].notna().all()                               # both years now have a spot price
print("Year 2 spot price added:", len(spot_y2), "intervals")
dfc.groupby("year")["rrp_kWh"].describe()[["min", "50%", "mean", "max"]].round(3)

Year 2 spot price added: 105120 intervals


,min,50%,mean,max
year,,,,
1,-1.0,0.134,0.183,15.5
2,-1.0,0.086,0.096,16.6


> **Finding 3.1 - Spot prices.** AEMO's timestamps *as published* (interval end) reproduce the site's January 2022 prices exactly (100% of 8,927 intervals, maximum difference 0), while shifting them to interval start matches only 22.7%. So the site's Year 1 prices are AEMO NSW1 prices on interval-end stamps, and the 2023 prices are aligned the same way. December 2022 is downloaded as well because the first Year 2 interval (00:00 on 1 January 2023) is the last AEMO interval of December 2022.
>
> 2023 prices were about half of 2022's: the mean fell from 183 to 96 AUD/MWh and the median from 134 to 86 AUD/MWh. 2022 was an energy-crisis year, with gas price spikes and coal outages culminating in AEMO suspending the market in June 2022. Negative prices more than doubled, from 3.0% of intervals in 2022 to 6.3% in 2023. The 2023 maximum (16,600 AUD/MWh on 4 September 2023) is a genuine price: the market price cap rose from 15,500 to 16,600 AUD/MWh on 1 July 2023.
>
> For the battery, this means a wholesale pass-through customer in 2023 earns less from price spikes than in 2022, but gets twice as many chances to charge cheaply (or be paid to charge) and to curtail exports when prices are negative.

## Task 3.2 - Wholesale pass-through price: decisions

The site metadata says the wholesale tariff is "taken directly from market spot price data". The brief adds that a pass-through customer also pays time-of-day network costs, market fees, environmental certificates and hedging costs. This matches how Amber Electric works: all of these are passed through at cost, and a monthly subscription ($25, as in the brief) is the only margin. The flat and ToU tariffs are used exactly as given.

**Import ($/kWh)** = [spot x loss factor + network(period, financial year) + environmental + market fees + hedging] x (1 + GST)
**Export ($/kWh)** = spot (negative prices included, so exporting at a negative price costs money and PV should curtail)

| Component | Value | Source / reason |
|---|---|---|
| Loss factor | x 1.034 (DLF 6.09%, MLF -2.52%) | ACIL Allen for the AER, DMO 2023-24, Essential NSLP (Table 4.33) |
| Network, 1 Jan-30 Jun 2023 | peak 16.12, shoulder 12.08, off-peak 4.90 c/kWh | Essential Energy BLNT3AL, indicative 2022-23 schedule (Mar 2022). *Limitation: indicative, not the final price list* |
| Network, 1 Jul-31 Dec 2023 | peak 17.64, shoulder 12.70, off-peak 5.05 c/kWh | Essential Energy BLNT3AL, 2023-24 network price list |
| Network time windows | the assignment's ToU windows (setting `NETWORK_WINDOWS`) | Keeps one definition of peak/shoulder/off-peak across all three tariffs. Essential's own windows differ (peak 5-8 pm weekdays); **tutor's advice requested**, switchable in Part 0 |
| Environmental schemes | 1.78 c/kWh | LRET 7.27 + SRES 6.86 + NSW ESS 3.26 + PDRS 0.42 $/MWh (ACIL Allen Tables 4.6-4.10) |
| Market fees | 0.16 c/kWh | AEMO fees 1.15 + ancillary services 0.42 $/MWh (Tables 4.11-4.12) |
| Hedging | 0.26 c/kWh | Prudential costs only (AEMO 0.77 + hedge 1.86 $/MWh). A pass-through customer is exposed to the spot price, so full hedge premiums do not apply |
| Daily charges | excluded from all three tariffs | The given flat and ToU tariffs have none; excluding them keeps the comparison like-for-like |
| GST | +10% on imports | The given flat and ToU rates are taken as GST-inclusive |

Only Year 2 (2023) is priced: it is the year the optimisation is run on.

In [449]:
def network_period(index: pd.DatetimeIndex, windows: str = NETWORK_WINDOWS) -> np.ndarray:
    """'peak' / 'shoulder' / 'offpeak' for each timestamp, by the hour of its stamp (as the site's ToU tariff does)."""
    h = index.hour
    if windows == "assignment":        # Finding 1.4: identical every day
        return np.select([(h >= 19) & (h < 23), (h >= 7) & (h < 19)], ["peak", "shoulder"], "offpeak")
    if windows == "essential":         # weekdays only; weekends all off-peak; applied on the data's AEST clock
        wk = index.dayofweek < 5
        return np.select([wk & (h >= 17) & (h < 20),
                          wk & (((h >= 7) & (h < 17)) | ((h >= 20) & (h < 22)))], ["peak", "shoulder"], "offpeak")
    raise ValueError(f"unknown NETWORK_WINDOWS: {windows!r}")


def financial_year(index: pd.DatetimeIndex) -> np.ndarray:
    """Australian financial year label for each timestamp, e.g. '2022-23' (the year starts 1 July)."""
    start = np.where(index.month >= 7, index.year, index.year - 1)
    return np.array([f"{y}-{str(y + 1)[-2:]}" for y in start])

In [450]:
"""Tests for Task 3.3 helpers"""
_t = pd.to_datetime(["2023-01-02 06:55", "2023-01-02 07:00", "2023-01-02 19:00", "2023-01-07 18:00"])  # Mon x3, Sat
assert network_period(_t, "assignment").tolist() == ["offpeak", "shoulder", "peak", "shoulder"]
assert network_period(_t, "essential").tolist()  == ["offpeak", "shoulder", "peak", "offpeak"]       # 19:00 Mon in 5-8 pm peak; weekend off-peak
assert financial_year(pd.to_datetime(["2023-06-30 23:55", "2023-07-01 00:05"])).tolist() == ["2022-23", "2023-24"]
print("Task 3.3 helper tests passed")

Task 3.3 helper tests passed


## Task 3.3 - Build the wholesale pass-through price
`wholesale_retail_price` turns the spot price into the import and export price a pass-through customer pays, using the Part 0 settings and the formula in Task 3.2. The tests below fix the expected answer for a few hand-calculated intervals, so they catch unit slips (c/kWh vs $/kWh) as well as logic errors.

In [451]:
def wholesale_retail_price(spot: pd.Series, windows: str = NETWORK_WINDOWS) -> pd.DataFrame:
    """Wholesale pass-through prices for each interval of `spot` ($/kWh, interval-end stamps).

    import = (spot x LOSS_FACTOR + network + environmental + fees + hedging) x (1 + GST)
    export = spot   (negative included)
    network comes from NETWORK_RATES[financial year][period]; all adders are in c/kWh, so divide by 100.
    Returns a DataFrame on spot's index with columns: period, network_kWh, import_kWh, export_kWh.
    """
    # period for each interval (network_period) and financial year (financial_year)
    period = network_period(spot.index, windows)
    fy = financial_year(spot.index)

    network = np.array([NETWORK_RATES[f][p] for f, p in zip(fy, period)]) / 100   # c/kWh -> $/kWh

    import_kwh = (spot * LOSS_FACTOR + network + (ENV_C_KWH + FEES_C_KWH + HEDGE_C_KWH) / 100) * (1 + GST)

    out = pd.DataFrame({"period" : period, "network_kWh": network,"import_kWh": import_kwh, "export_kWh": spot }, index=spot.index)
    return out

In [452]:
"""Tests for Task 3.3 (hand-calculated)"""
_idx = pd.to_datetime(["2023-01-10 02:00",    # off-peak, FY 2022-23
                       "2023-08-15 19:30",    # peak, FY 2023-24 (assignment windows)
                       "2023-08-15 12:00"])   # shoulder, FY 2023-24, negative spot
_w = wholesale_retail_price(pd.Series([0.05, 0.10, -0.04], index=_idx), windows="assignment")
_adders = (1.781 + 0.157 + 0.263) / 100                                        # 0.02201 $/kWh
assert _w["period"].tolist() == ["offpeak", "peak", "shoulder"]
assert np.allclose(_w["network_kWh"], [0.048957, 0.176443, 0.126958])
assert np.isclose(_w["import_kWh"].iloc[0], (0.05 * 1.034 + 0.048957 + _adders) * 1.1)   # = 0.13493
assert np.isclose(_w["import_kWh"].iloc[1], (0.10 * 1.034 + 0.176443 + _adders) * 1.1)   # = 0.33204
assert np.isclose(_w["import_kWh"].iloc[2], (-0.04 * 1.034 + 0.126958 + _adders) * 1.1)  # negative spot lowers the import price
assert _w["export_kWh"].tolist() == [0.05, 0.10, -0.04]                                  # export = spot, negatives kept
assert _w.index.equals(_idx)
print("Task 3.3 tests passed")

Task 3.3 tests passed


In [453]:
wholesale = wholesale_retail_price(dfc.loc[dfc.year == 2, "rrp_kWh"])
print("Network windows used:", NETWORK_WINDOWS)
wholesale.groupby("period")[["network_kWh", "import_kWh", "export_kWh"]].mean().round(3)

Network windows used: assignment


,network_kWh,import_kWh,export_kWh
period,,,
offpeak,0.050,0.183,0.091
peak,0.169,0.343,0.117
shoulder,0.124,0.265,0.092


## Task 3.4 - One price table for all three tariffs
`prices` holds the import and export price ($/kWh) for flat, ToU and wholesale at every Year 2 interval. From Part 6 onwards every bill and every optimisation reads from this one table, so a tariff is just a pair of columns: the same code runs all three.

In [454]:
TARIFFS = ["flat", "tou", "wholesale"]
y2 = dfc.year == 2
prices = pd.DataFrame({
    "flat_imp":      dfc.loc[y2, "flat_imp"],
    "flat_exp":      dfc.loc[y2, "flat_exp"],
    "tou_imp":       dfc.loc[y2, "tou_imp"],
    "tou_exp":       dfc.loc[y2, "tou_exp"],
    "wholesale_imp": wholesale["import_kWh"],
    "wholesale_exp": wholesale["export_kWh"],
})

# Contract checks: complete, Year 2 only, one row per interval
assert prices.notna().all().all() and len(prices) == y2.sum()
assert prices.index.equals(dfc.index[y2])
assert (prices[["flat_imp", "tou_imp"]] > prices[["flat_exp", "tou_exp"]].values).all().all()   # fixed tariffs: import > export
print("prices:", prices.shape)
prices.describe().loc[["min", "50%", "mean", "max"]].round(3)

prices: (105120, 6)


,flat_imp,flat_exp,tou_imp,tou_exp,wholesale_imp,wholesale_exp
min,0.3,0.05,0.150,0.05,-0.980,-1.000
50%,0.3,0.05,0.250,0.05,0.228,0.086
mean,0.3,0.05,0.242,0.05,0.251,0.096
max,0.3,0.05,0.400,0.05,19.045,16.600


## Task 3.5 - How the three tariffs compare through the day

In [455]:
# Figure 3.1 - Average price by time of day, Year 2 (import solid, export dashed)
by_hour = prices.groupby(prices.index.hour + prices.index.minute / 60).mean()
colours = {"flat": "#7f7f7f", "tou": "#1f77b4", "wholesale": "#d62728"}
names = {"flat": "Flat", "tou": "Time of use", "wholesale": "Wholesale pass-through"}

fig = go.Figure()
for t in TARIFFS:
    fig.add_trace(go.Scatter(x=by_hour.index, y=by_hour[f"{t}_imp"] * 100, name=f"{names[t]} - import",
                             line=dict(color=colours[t], width=2.5, shape="hv" if t != "wholesale" else "linear")))
    fig.add_trace(go.Scatter(x=by_hour.index, y=by_hour[f"{t}_exp"] * 100, name=f"{names[t]} - export",
                             line=dict(color=colours[t], width=1.5, dash="dash", shape="hv" if t != "wholesale" else "linear")))
fig.update_layout(
    title="Average price by time of day (2023): wholesale is cheapest at midday but dearest 4-7 pm; ToU is dearest 7-11 pm",
    xaxis=dict(title="Hour of day", tickmode="array", tickvals=list(range(0, 25, 3))),
    yaxis_title="Price (c/kWh)", template="plotly_white", height=460,
    legend=dict(orientation="h", yanchor="top", y=-0.18, x=0))
fig.show()

# Numbers behind the figure: average import price in four parts of the day
blocks = pd.cut(prices.index.hour, [0, 7, 10, 16, 19, 23, 24], right=False,
                labels=["00-07", "07-10", "10-16 (solar)", "16-19", "19-23 (peak)", "23-24"])
(prices.groupby(blocks, observed=True)[[f"{t}_imp" for t in TARIFFS] + ["wholesale_exp"]].mean() * 100).round(1)

,flat_imp,tou_imp,wholesale_imp,wholesale_exp
00-07,30.0,15.0,18.1,9.0
07-10,30.0,25.0,23.4,6.4
10-16 (solar),30.0,25.0,21.8,5.1
16-19,30.0,25.0,39.2,20.3
19-23 (peak),30.0,40.0,34.3,11.7
23-24,30.0,15.0,19.3,10.1


> **Finding 3.2-3.5 - Tariffs.** TODO in my words: (1) the wholesale price formula and its sources (Task 3.2), and that the network windows follow the assignment's ToU windows pending tutor advice; (2) how wholesale compares with ToU by time of day (table above): cheaper or dearer in each block, and where the export price beats 5c; (3) what that means for the battery on each tariff (e.g. flat rewards only self-consumption; ToU rewards shifting into 19-23; wholesale rewards charging at midday and avoiding/selling into evening spikes).

# Part 4 - Site Characterisation

Part 4 answers the questions the battery design depends on: is the clock what we think it is, how much energy could a battery actually move each day, how big is the PV system, and how is the battery connected? Everything uses Year 2 (2023), the year the optimisation is run on, unless stated.

## Task 4.1 - Clock check: where is solar noon?

### Background
Finding 1.2 showed no daylight-saving jumps, which suggests the data runs on AEST all year. Solar noon gives an independent test. Thredbo is at about 148.3°E, 1.7° west of the AEST reference meridian (150°E), so mean solar noon is about 12:07 AEST. The equation of time moves it by up to about ±16 minutes across the year. If the data were on local daylight-saving time, the centre of PV generation would jump by an hour, to around 13:00, between October and April.

`pv_centroid_hour` finds the "centre of mass" of daily generation for each month: the PV-weighted mean hour of day.

In [456]:
def pv_centroid_hour(pv: pd.Series) -> pd.Series:
    """PV-weighted mean hour of day for each month (e.g. 12.5 = 12:30): the centre of daily generation.

    centroid = sum(hour x pv) / sum(pv), grouped by calendar month, using fractional hours (hour + minute / 60).
    """
    hour = pv.index.hour + pv.index.minute / 60.0
    centroid = (pv * hour).groupby(pv.index.month).sum()/ pv.groupby(pv.index.month).sum()

    return centroid


In [457]:
"""Tests for Task 4.1"""
_i = pd.date_range("2023-01-01", "2023-02-28 23:55", freq="5min")
_h = np.asarray(_i.hour + _i.minute / 60)
_pv = pd.Series(np.clip(1 - np.abs(_h - 12) / 6, 0, None), index=_i)                    # symmetric about 12:00
_pv[_i.month == 2] = np.clip(1 - np.abs(_h[_i.month == 2] - 13) / 6, 0, None)          # Feb shifted 1 h (like daylight saving)
_c = pv_centroid_hour(_pv)
assert np.isclose(_c.loc[1], 12.0) and np.isclose(_c.loc[2], 13.0)
print("Task 4.1 tests passed")

Task 4.1 tests passed


In [458]:
centroid = pd.DataFrame({
    "Y1 measured PV":   pv_centroid_hour(dfc.loc[dfc.year == 1, "pv_meas_kW"].dropna()),
    "Y2 commercial PV": pv_centroid_hour(dfc.loc[dfc.year == 2, "pv_comm_kW"]),
})
print("Centre of PV generation, minutes after 12:00 (AEST if the clock is standard time):")
((centroid - 12) * 60).round(0).T

Centre of PV generation, minutes after 12:00 (AEST if the clock is standard time):


timestamp,1,2,3,4,5,6,7,8,9,10,11,12
Y1 measured PV,37.0,40.0,37.0,33.0,34.0,36.0,44.0,43.0,29.0,17.0,14.0,20.0
Y2 commercial PV,26.0,44.0,43.0,31.0,33.0,40.0,43.0,38.0,27.0,17.0,15.0,24.0


> **Finding 4.1 - Clock.** TODO in my words: the centre of generation each month (minutes after 12:00); whether there is a one-hour jump in Oct-Apr (daylight saving) or not; so is the data AEST all year? (Small offsets from 12:07 can come from the equation of time, panel orientation and the interval-end timestamps.)

## Task 4.2 - Daily energy balance: how much could a battery shift?

### Background
Without a battery, each 5-minute interval splits into three flows (kW):
- **self-consumed** = min(load, PV): solar used directly;
- **export** = PV − load when PV exceeds load: surplus solar sent to the grid;
- **import** = load − PV when load exceeds PV: demand met from the grid.

Summed over a day (kW × `dt_h` = kWh), **shiftable = min(export, import)** is the most energy a perfect battery could move from surplus to deficit that day. It is an upper bound for sizing: it ignores losses, power limits and carrying charge to the next day.

In [459]:
def daily_energy_balance(load: pd.Series, pv: pd.Series, dt_h: float = DT_H) -> pd.DataFrame:
    """Daily energy with no battery (kWh): load, pv, self_consumed, export, import, shiftable.

    Per interval (kW): self_consumed = min(load, pv); export = surplus PV; import = unmet load.
    Convert to kWh with dt_h, sum each day, then shiftable = min(export, import) for each day.
    """
    self_consumed = np.minimum(load, pv)
    export = (pv - load).clip(lower = 0)
    imp = (load - pv).clip(lower = 0)

    flows = pd.DataFrame({
        "load": load,
        "pv": pv,
        "self_consumed": self_consumed,
        "export": export,
        "import": imp
    }) * dt_h

    # Resample and sum to get daily totals
    daily = flows.resample("D").sum()
    
    # Shiftable energy is the minimum of daily export and daily import
    daily["shiftable"] = np.minimum(daily["export"], daily["import"])

    return daily

In [460]:
"""Tests for Task 4.2"""
_i = pd.to_datetime(["2023-01-01 10:00", "2023-01-01 11:00", "2023-01-01 12:00", "2023-01-01 13:00",
                     "2023-01-02 10:00", "2023-01-02 11:00", "2023-01-02 12:00", "2023-01-02 13:00"])
_load = pd.Series([1, 1, 3, 2,  2, 2, 2, 2], index=_i, dtype=float)
_pv   = pd.Series([0, 4, 1, 0,  0, 0, 0, 0], index=_i, dtype=float)
_b = daily_energy_balance(_load, _pv, dt_h=1.0)
assert _b.loc["2023-01-01"].to_dict() == dict(load=7, pv=5, self_consumed=2, export=3, **{"import": 5}, shiftable=3)
assert _b.loc["2023-01-02", "shiftable"] == 0                                   # no surplus solar -> nothing to shift
assert np.allclose(_b["load"], _b["self_consumed"] + _b["import"])              # energy balance: load side
assert np.allclose(_b["pv"],   _b["self_consumed"] + _b["export"])              # energy balance: PV side
print("Task 4.2 tests passed")

Task 4.2 tests passed


In [461]:
bal = daily_energy_balance(dfc.loc[dfc.year == 2, "load_kW"], dfc.loc[dfc.year == 2, "pv_kW"])
tot = bal.sum()
print(f"No battery, 2023: self-consumption {tot.self_consumed / tot.pv:.0%} of PV is used on site; "
      f"self-sufficiency {tot.self_consumed / tot.load:.0%} of load is met by PV")
print(f"Shiftable energy per day: median {bal.shiftable.median():.1f} kWh, 90th percentile {bal.shiftable.quantile(0.9):.1f} kWh, max {bal.shiftable.max():.1f} kWh")
monthly = bal.groupby(bal.index.month).mean()
monthly.round(1)

No battery, 2023: self-consumption 54% of PV is used on site; self-sufficiency 65% of load is met by PV
Shiftable energy per day: median 7.4 kWh, 90th percentile 10.7 kWh, max 14.2 kWh


,load,pv,self_consumed,export,import,shiftable
timestamp,,,,,,
1,32.3,48.5,24.0,24.4,8.3,7.7
2,35.1,49.0,26.7,22.2,8.3,7.4
3,28.9,36.4,19.6,16.8,9.3,8.1
4,22.8,26.0,13.2,12.8,9.6,7.3
5,24.2,19.4,10.8,8.6,13.4,7.4
6,22.1,12.6,7.2,5.4,14.9,5.2
7,23.8,18.5,9.8,8.7,14.0,7.4
8,23.9,24.1,13.3,10.8,10.6,8.2
9,34.9,40.5,26.8,13.7,8.1,6.9


In [462]:
# Figure 4.1 - Monthly average daily energy (2023) and what a battery could shift
months = pd.to_datetime(monthly.index.astype(str), format="%m").strftime("%b")
fig = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38], horizontal_spacing=0.1,
                    subplot_titles=("Average day in each month (kWh/day)", "Shiftable energy per day (all 365 days)"))
for col, name, colour in [("load", "Load", "#444444"), ("pv", "PV generation", "#f2a900"),
                          ("export", "Export (surplus PV)", "#1f77b4"), ("import", "Import (unmet load)", "#d62728")]:
    fig.add_trace(go.Bar(x=months, y=monthly[col], name=name, marker_color=colour), row=1, col=1)
fig.add_trace(go.Scatter(x=months, y=monthly["shiftable"], name="Shiftable (battery upper bound)",
                         mode="lines+markers", line=dict(color="#2ca02c", width=3)), row=1, col=1)
fig.add_trace(go.Histogram(x=bal["shiftable"], nbinsx=30, marker_color="#2ca02c", opacity=0.8, showlegend=False), row=1, col=2)
for q, dash, label, side in [(0.5, "dash", "Median", "top left"), (0.9, "dot", "90th pct", "top right")]:
    v = bal["shiftable"].quantile(q)
    fig.add_vline(x=v, line_dash=dash, line_color="black", row=1, col=2,
                  annotation_text=f"{label}<br>{v:.1f} kWh", annotation_position=side)
fig.update_xaxes(title_text="kWh/day", row=1, col=2)
fig.update_yaxes(title_text="kWh/day", row=1, col=1)
fig.update_yaxes(title_text="Number of days", row=1, col=2)
fig.update_layout(title="Load, PV and shiftable energy, 2023", barmode="group", template="plotly_white", height=460,
                  legend=dict(orientation="h", yanchor="top", y=-0.15, x=0))
fig.show()

> **Finding 4.2 - Energy balance.** TODO in my words: no-battery self-consumption and self-sufficiency; how PV and load swap seasonally (surplus in summer, shortfall in winter); and the key sizing number: shiftable energy per day (median, 90th percentile) and how steady it is across months. What battery size (usable kWh) does that point to for self-consumption, and why might ToU/wholesale arbitrage justify more (tested in Part 8)?

## Task 4.3 - PV system size

The metadata gives the performance ratio (85%) but not the array size. It can be estimated from data already in hand: the median daily PV-to-irradiance ratio (Task 1.6) is the array's effective output in kW per kW/m² of sunlight, which equals size (kWp) × performance ratio. Compare the estimate with the measured peak output (10.31 kW, Finding 1.3).

In [463]:
# Task 4.3 - Estimate PV array size (kWp)
# TODO: use the median of the Year 1 measured ratio from Task 1.6 (ratio["Y1 measured"]) and PV_PR from Part 0
PV_KWP_EST = ratio["Y1 measured"].median() / PV_PR
print(f"Estimated PV array size: {PV_KWP_EST:.3f} kWp")

Estimated PV array size: 8.904 kWp


## Task 4.4 - Battery topology: AC- or DC-coupled?

The brief asks for an assumption on how the battery connects, because it decides which conversion losses apply.

| | **AC-coupled** | **DC-coupled (hybrid inverter)** |
|---|---|---|
| How it connects | Battery has its own inverter on the house AC side, next to the existing PV inverter | PV and battery share one hybrid inverter on a DC bus |
| Solar to battery | PV DC -> AC (PV inverter) -> DC (battery inverter): **two** conversions | PV DC -> battery DC: **no** inverter conversion |
| Battery to house | DC -> AC: one conversion | DC -> AC: one conversion |
| Grid to battery (ToU / wholesale arbitrage) | AC -> DC: one conversion | AC -> DC: one conversion |
| Typical use | Retrofit to an existing PV system | New PV + battery installed together |
| Product examples | Separate AC battery systems | Hybrid-inverter battery systems |

Efficiency values for the chosen topology and product are set and sourced in Part 5 (battery model).

**Decision (TODO, mine):** which topology fits this site, and why? Consider: the site already has a ~10 kW PV system (retrofit or new build?); how much of the battery's charging comes from solar vs the grid on each tariff (DC coupling saves a conversion only on solar charging); and the cost and simplicity of each option.

In [464]:
TOPOLOGY = "AC"   # TODO: "AC" or "DC" - my decision from Task 4.4, used by the battery model in Part 5

> **Finding 4.3-4.4 - System.** TODO in my words: estimated PV size and how it compares with the measured peak; the chosen topology and the reason.

# Part 5 - Battery Model

Part 5 turns a real product's datasheet into the battery model every later Part uses: how much energy it holds, how fast it charges and discharges, what is lost on the way in and out, what limits keep it healthy, and what each kWh of cycling costs in wear. The topology is **AC-coupled** (Task 4.4): the battery is retrofitted next to the existing PV inverter, so efficiencies are AC-to-AC.

## Task 5.1 - Choose the product

Sizing evidence from Task 4.2: shiftable energy is about **7.4 kWh/day (median)** and **10.7 kWh/day (90th percentile)**, fairly steady across the year. ToU and wholesale arbitrage may justify more (tested in Part 8).

Two current products that can be AC-coupled to an existing PV system were shortlisted. (A standard Powerwall 3 was ruled out: it can only AC-couple to existing PV inverters of 5 kW or less per unit, and this site's array is about 9 kWp.)

| | **Tesla Powerwall 3AC** | **Sigenergy SigenStor** (AC-coupled, single phase) |
|---|---|---|
| Usable energy | 13.5 kWh | 7.8 kWh per 8 kWh module (2 modules = 15.6 kWh) |
| Continuous power | 5 kW charge; 5 kW output | 4 kW per module; system limited by the 5 kW energy controller (SP 16: 5 kW) |
| Efficiency | 89% battery-to-home/grid (25 °C, beginning of life, 3.3 kW) | Not published AC-to-AC. Energy controller max 97.6%; AC coupling adds one conversion. **Assumption needed** |
| Chemistry | not stated on the 3AC datasheet | LFP, cycle life up to 10,000 |
| Operating temperature | -20 to 50 °C (may derate above 40 °C) | -20 to 55 °C (battery) |
| Warranty | 10 years (retention and throughput not stated) | 10 years / 70%; minimum throughput 23.77 MWh per 8 kWh module |
| Indicative installed price (before rebate) | about $13,360 for 13.5 kWh (2026 average) | about $1,000 per usable kWh (16 kWh about $14,000-17,000) |

**Points to weigh:** 13.5 kWh (Tesla) or 15.6 kWh (two Sigen modules) both exceed the 10.7 kWh 90th percentile, so the extra room is only useful for ToU/wholesale arbitrage; one Sigen module (7.8 kWh) matches the median. Sigenergy's throughput warranty makes the degradation cost in Task 5.3 directly calculable (23.77 MWh / 7.8 kWh = about 3,050 equivalent full cycles to 70%); Tesla's 3AC sheet gives no throughput, so its wear cost would need an assumption. Prices are 2025-26 installed prices (the battery would be bought now, not in 2023); the federal Cheaper Home Batteries rebate (from July 2025) is a sensitivity for Part 8.

**Decision (TODO, mine):** which product and how many modules, and why.

Sources: Tesla Powerwall 3AC datasheet (AU, 2026); SolarQuotes Powerwall 3 review (AC-coupling limit); Sigenergy SigenStor and Sigen Battery datasheets; SigenStor factory warranty (throughput); Gridly / Solar Choice / Aussie Solar Tech price surveys. Full references in the report.

In [465]:
@dataclass(frozen=True)
class BatterySpec:
    """Everything the battery model needs, in one place. Energy in kWh, power in kW, fractions 0-1."""
    name: str
    usable_kWh: float            # usable energy at beginning of life
    p_charge_kW: float           # max continuous charge power (AC side)
    p_discharge_kW: float        # max continuous discharge power (AC side)
    rte: float                   # round-trip efficiency, AC to AC
    soc_min: float = 0.0         # operating window as a fraction of usable energy
    soc_max: float = 1.0
    soc_init: float = 0.5        # state of charge at the start of a simulation
    temp_opt_low: float = 0.0    # datasheet optimum temperature range (degC); full power inside it
    temp_opt_high: float = 30.0
    power_frac_outside: float = 1.0   # fraction of max power allowed outside the optimum range (assumption)
    warranty_cycles: float = np.nan   # full equivalent cycles to end of warranty
    eol_capacity: float = np.nan      # capacity fraction remaining at end of warranty
    capex_aud: float = np.nan         # installed cost (AUD), for degradation cost and payback

    @property
    def eta_ch(self) -> float:   # one-way efficiency: split the round trip evenly between charge and discharge
        return float(np.sqrt(self.rte))

    @property
    def eta_dis(self) -> float:
        return float(np.sqrt(self.rte))

    @property
    def e_min(self) -> float:
        return self.soc_min * self.usable_kWh

    @property
    def e_max(self) -> float:
        return self.soc_max * self.usable_kWh

In [466]:
"""Tests for BatterySpec"""
_b = BatterySpec("test", usable_kWh=10, p_charge_kW=5, p_discharge_kW=5, rte=0.81, soc_min=0.1, soc_max=0.9)
assert np.isclose(_b.eta_ch, 0.9) and np.isclose(_b.eta_ch * _b.eta_dis, 0.81)   # one-way efficiencies multiply to the round trip
assert np.isclose(_b.e_min, 1.0) and np.isclose(_b.e_max, 9.0)
print("BatterySpec tests passed")

BatterySpec tests passed


In [ ]:
# Task 5.1 - Battery spec. Assumptions marked (A) to confirm before final results.
SIGEN = BatterySpec(
    name="Sigenergy SigenStor 2 x 8 kWh", usable_kWh=15.6, p_charge_kW=5, p_discharge_kW=5,
    rte=0.90,                              # (A) no AC-to-AC figure published; in line with similar AC batteries
    power_frac_outside=0.5,                # (A) half power below 0 C
    warranty_cycles=47540 / 15.6, eol_capacity=0.70,   # throughput warranty 23.77 MWh per module, to 70%
    capex_aud=15600,                       # about $1,000 per usable kWh, before rebate
)
BATTERY = SIGEN
BATTERY

## Task 5.2 - Operating limits: state of charge and temperature

### Background
**State of charge window.** The datasheet's usable energy is already the safe range set by the manufacturer. A tighter window (e.g. keeping 10-20% in reserve) can be chosen for backup or longevity, at the cost of less energy to shift.

**Temperature.** Thredbo ranges from about -10 to 26 °C (Finding 1.3), so the battery spends many winter hours below the datasheets' optimum range of 0-30 °C. Neither datasheet publishes a derating curve, so this is a stated assumption: inside the optimum range the battery runs at full power; outside it, power is limited to a fraction `power_frac_outside`. This reduces wear in cold conditions, as the brief asks. The air temperature from the site data is used as a conservative stand-in for the battery's own temperature (an indoor or garage install would run warmer).

`power_limit_factor` returns the fraction of rated power allowed in each interval.

In [468]:
def power_limit_factor(temp_C: pd.Series, spec: BatterySpec) -> pd.Series:
    """Fraction of rated charge/discharge power allowed at each temperature.

    1.0 inside [spec.temp_opt_low, spec.temp_opt_high]; spec.power_frac_outside outside it.
    """
    out = pd.Series(np.where((temp_C >= spec.temp_opt_low) & (temp_C <= spec.temp_opt_high), 1.0, spec.power_frac_outside), index=temp_C.index)
    return out

In [469]:
"""Tests for Task 5.2"""
_s = BatterySpec("t", 10, 5, 5, 0.9, temp_opt_low=0, temp_opt_high=30, power_frac_outside=0.5)
_f = power_limit_factor(pd.Series([-5.0, 0.0, 15.0, 30.0, 35.0]), _s)
assert _f.tolist() == [0.5, 1.0, 1.0, 1.0, 0.5]                     # the boundaries count as inside
assert isinstance(_f, pd.Series)
print("Task 5.2 tests passed")

Task 5.2 tests passed


In [470]:
# How often would the limit bite at this site? (Year 2, using my BATTERY settings)
_factor = power_limit_factor(dfc.loc[dfc.year == 2, "temp_C"], BATTERY)
print(f"Share of 2023 intervals outside the optimum range: {(_factor < 1).mean():.1%}")
(_factor < 1).groupby(_factor.index.month).mean().mul(100).round(0).rename("% of intervals derated, by month")

Share of 2023 intervals outside the optimum range: 0.0%


timestamp
1     0.0
2     0.0
3     0.0
4     0.0
5     0.0
6     0.0
7     0.0
8     0.0
9     0.0
10    0.0
11    0.0
12    0.0
Name: % of intervals derated, by month, dtype: float64

## Task 5.3 - Degradation cost

### Background
Every kWh cycled through the battery uses up part of its life. Putting a price on that wear stops the optimiser cycling the battery for tiny gains (the brief asks for this). A standard estimate spreads the installed cost over all the energy the battery will discharge before end of warranty:

**degradation cost ($/kWh discharged) = capex / (warranty cycles × usable kWh × average capacity over life)**

Capacity fades roughly linearly from 100% to the end-of-warranty value, so the average capacity is (1 + eol_capacity) / 2. The cost is applied to each kWh discharged in the optimiser (Part 7).

In [471]:
def degradation_cost_per_kwh(spec: BatterySpec) -> float:
    """Wear cost ($ per kWh discharged) = capex / (warranty_cycles x usable_kWh x average capacity over life)."""
    avg_capacity = (1.0 + spec.eol_capacity) / 2.0
    deg_cost = spec.capex_aud / (spec.warranty_cycles * spec.usable_kWh * avg_capacity)

    return deg_cost

In [472]:
"""Tests for Task 5.3"""
_s = BatterySpec("t", usable_kWh=10, p_charge_kW=5, p_discharge_kW=5, rte=0.9,
                 warranty_cycles=6000, eol_capacity=0.6, capex_aud=12000)
assert np.isclose(degradation_cost_per_kwh(_s), 12000 / (6000 * 10 * 0.8))   # = $0.25/kWh
print("Task 5.3 tests passed")

Task 5.3 tests passed


**TODO (mine):** fill `warranty_cycles`, `eol_capacity` and `capex_aud` in `BATTERY` (Task 5.1 cell). The capex needs a source: an installed price for the chosen product in 2023 (and whether to subtract any rebate). Then compare the degradation cost with the price spreads from Task 3.5: a shift is only worth doing if the spread beats the wear cost plus the round-trip loss.

In [473]:
print(f"{BATTERY.name}: degradation cost {degradation_cost_per_kwh(BATTERY):.3f} $/kWh discharged")

TODO: degradation cost nan $/kWh discharged


## Task 5.4 - State-of-charge simulator

### Background
Given a battery power schedule (kW, **positive = discharging, negative = charging**, as in Part 0), the stored energy evolves as:
- charging at p kW for dt_h hours adds **eta_ch × p × dt_h** kWh (some energy is lost going in);
- discharging at p kW removes **p / eta_dis × dt_h** kWh (more is drawn from storage than is delivered).

`simulate_soc` runs this forward. It is used to check that any schedule (rule-based in Part 6, optimised in Parts 7 and 11) respects the battery's limits, independently of the optimiser that produced it.

In [474]:
def simulate_soc(p_batt_kW: pd.Series, spec: BatterySpec, dt_h: float = DT_H, e0_kWh: float = None) -> pd.Series:
    """Stored energy (kWh) at the END of each interval for a battery power schedule.

    p_batt_kW: positive = discharging, negative = charging. e0_kWh defaults to spec.soc_init x usable_kWh.
    Does not clip: if the schedule breaks a limit, the result shows it (checked by the caller).
    """
    e0 = spec.soc_init * spec.usable_kWh if e0_kWh is None else e0_kWh

    # Charging (p < 0) adds eta_ch * |p| * dt_h; discharging (p > 0) removes (p / eta_dis) * dt_h
    delta_e = pd.Series(
        np.where(
            p_batt_kW < 0,
            spec.eta_ch * (-p_batt_kW) * dt_h,
            -(p_batt_kW / spec.eta_dis) * dt_h,
        ),
        index=p_batt_kW.index,
    )

    return e0 + delta_e.cumsum()

In [475]:
"""Tests for Task 5.4"""
_s = BatterySpec("t", usable_kWh=10, p_charge_kW=5, p_discharge_kW=5, rte=0.81)        # eta = 0.9 each way
_p = pd.Series([-2.0, -2.0, 0.0, 3.0])                                                   # charge, charge, idle, discharge
_e = simulate_soc(_p, _s, dt_h=1.0, e0_kWh=5.0)
assert np.allclose(_e, [5 + 1.8, 5 + 3.6, 5 + 3.6, 5 + 3.6 - 3 / 0.9])
# a full round trip loses exactly (1 - rte) of the energy charged from the grid
_e2 = simulate_soc(pd.Series([-1.0, 0.81]), _s, dt_h=1.0, e0_kWh=0.0)
assert np.isclose(_e2.iloc[-1], 0.0)                                                     # 1 kWh in -> 0.81 kWh out
print("Task 5.4 tests passed")

Task 5.4 tests passed


> **Finding 5.1-5.4 - Battery model.** TODO in my words: the chosen product and size (why it fits the 7.4 / 10.7 kWh/day evidence); efficiency and SOC window; the temperature assumption and how often it bites at Thredbo; the degradation cost ($/kWh) and what spread a shift needs to be worth it.